In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import torch

DATA_DIR = Path("/content/drive/MyDrive/greek-news-nlp")

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

required = [
    DATA_DIR / "el_val.jsonl",
    DATA_DIR / "greek-bert-news-classifier" / "model.safetensors",
    DATA_DIR / "qlora_qwen2.5-1.5b_ep2" / "adapter_config.json",
    DATA_DIR / "qlora_qwen2.5-1.5b_ep2" / "adapter_model.safetensors",
]

for path in required:
    print("OK" if path.is_file() else "MISSING", "—", path.name)

Mounted at /content/drive
CUDA available: True
GPU: Tesla T4
OK — el_val.jsonl
OK — model.safetensors
OK — adapter_config.json
OK — adapter_model.safetensors


In [ ]:
!nvidia-smi

import torch
print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

Wed Oct 14 12:33:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# CELL 3 — GPU and Model Files Verification

from pathlib import Path
from google.colab import drive
import torch
import json
import sys
import importlib.metadata as metadata

# ------------------------------------
# 1. Google Drive
# ------------------------------------

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

DATA_DIR = Path(
    "/content/drive/MyDrive/greek-news-nlp"
)

assert DATA_DIR.exists(), "Project directory not found"

# ------------------------------------
# 2. Required files
# ------------------------------------

required_files = {
    "Validation dataset":
        DATA_DIR / "el_val.jsonl",

    "GreekBERT weights":
        DATA_DIR / "greek-bert-news-classifier"
        / "model.safetensors",

    "QLoRA adapter configuration":
        DATA_DIR / "qlora_qwen2.5-1.5b_ep2"
        / "adapter_config.json",

    "QLoRA adapter weights":
        DATA_DIR / "qlora_qwen2.5-1.5b_ep2"
        / "adapter_model.safetensors",
}

print("REQUIRED FILES")
print("-" * 50)

for name, path in required_files.items():
    exists = path.is_file()
    print(f"{name}: {'FOUND' if exists else 'MISSING'}")
    print(" ", path)

assert all(
    path.is_file()
    for path in required_files.values()
), "Some required files are missing"

# ------------------------------------
# 3. Validation dataset
# ------------------------------------

with open(
    DATA_DIR / "el_val.jsonl",
    encoding="utf-8"
) as f:
    val_data = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

assert len(val_data) == 787

assert all(
    all(
        field in record
        for field in (
            "document_id",
            "text_clean",
            "GPT-IPTC-label",
        )
    )
    for record in val_data
)

assert len({
    record["document_id"]
    for record in val_data
}) == 787

print("\nVALIDATION DATASET")
print("-" * 50)
print("Articles:", len(val_data))
print("Unique IDs:", len({
    r["document_id"] for r in val_data
}))

# ------------------------------------
# 4. CUDA and hardware
# ------------------------------------

assert torch.cuda.is_available(), (
    "CUDA is not available"
)

device = torch.device("cuda:0")

gpu_properties = torch.cuda.get_device_properties(0)

print("\nGPU")
print("-" * 50)

print("GPU:", torch.cuda.get_device_name(0))
print(
    "Total VRAM:",
    round(gpu_properties.total_memory / 1024**3, 2),
    "GB"
)
print("CUDA build:", torch.version.cuda)

# ------------------------------------
# 5. Software versions
# ------------------------------------

print("\nSOFTWARE VERSIONS")
print("-" * 50)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)

packages = [
    "transformers",
    "accelerate",
    "peft",
    "bitsandbytes",
    "safetensors",
]

for package in packages:
    try:
        version = metadata.version(package)
    except metadata.PackageNotFoundError:
        version = "NOT INSTALLED"

    print(f"{package}: {version}")

print("\nOK — GPU and benchmark files verified")

REQUIRED FILES
--------------------------------------------------
Validation dataset: FOUND
  /content/drive/MyDrive/greek-news-nlp/el_val.jsonl
GreekBERT weights: FOUND
  /content/drive/MyDrive/greek-news-nlp/greek-bert-news-classifier/model.safetensors
QLoRA adapter configuration: FOUND
  /content/drive/MyDrive/greek-news-nlp/qlora_qwen2.5-1.5b_ep2/adapter_config.json
QLoRA adapter weights: FOUND
  /content/drive/MyDrive/greek-news-nlp/qlora_qwen2.5-1.5b_ep2/adapter_model.safetensors

VALIDATION DATASET
--------------------------------------------------
Articles: 787
Unique IDs: 787

GPU
--------------------------------------------------
GPU: Tesla T4
Total VRAM: 14.56 GB
CUDA build: 13.0

SOFTWARE VERSIONS
--------------------------------------------------
Python: 3.13.15
PyTorch: 2.11.0+cu130
transformers: 5.18.0
accelerate: 1.15.0
peft: 0.21.1
bitsandbytes: NOT INSTALLED
safetensors: 0.8.0

OK — GPU and benchmark files verified


In [ ]:
# CELL 4 — Frozen Inference Benchmark Protocol

import random
import hashlib
import json
from pathlib import Path

# ------------------------------------
# 1. Benchmark configuration
# ------------------------------------

SEED = 42
N_WARMUP = 3
N_MEASURED = 64
BATCH_SIZE = 1

BENCHMARK_DIR = (
    DATA_DIR / "results" / "inference_benchmark"
)

BENCHMARK_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# ------------------------------------
# 2. Select validation articles
# ------------------------------------

assert len(val_data) == 787

rng = random.Random(SEED)

indices = list(range(len(val_data)))
rng.shuffle(indices)

warmup_indices = indices[:N_WARMUP]

measured_indices = indices[
    N_WARMUP:N_WARMUP + N_MEASURED
]

warmup_data = [
    val_data[i] for i in warmup_indices
]

benchmark_data = [
    val_data[i] for i in measured_indices
]

# ------------------------------------
# 3. Validate selection
# ------------------------------------

warmup_ids = {
    row["document_id"]
    for row in warmup_data
}

benchmark_ids = {
    row["document_id"]
    for row in benchmark_data
}

assert len(warmup_data) == N_WARMUP
assert len(benchmark_data) == N_MEASURED

assert len(warmup_ids) == N_WARMUP
assert len(benchmark_ids) == N_MEASURED

assert warmup_ids.isdisjoint(
    benchmark_ids
)

assert all(
    row["text_clean"].strip()
    for row in benchmark_data
)

# ------------------------------------
# 4. Record dataset SHA256
# ------------------------------------

validation_path = DATA_DIR / "el_val.jsonl"

validation_sha256 = hashlib.sha256(
    validation_path.read_bytes()
).hexdigest()

# ------------------------------------
# 5. Create protocol manifest
# ------------------------------------

protocol = {
    "benchmark": "controlled_inference_benchmark_v1",
    "seed": SEED,
    "dataset": "validation",
    "validation_sha256": validation_sha256,
    "warmup_samples": N_WARMUP,
    "measured_samples": N_MEASURED,
    "batch_size": BATCH_SIZE,
    "warmup_document_ids": [
        row["document_id"]
        for row in warmup_data
    ],
    "measured_document_ids": [
        row["document_id"]
        for row in benchmark_data
    ],
    "models": [
        "GreekBERT fine-tuned",
        "Qwen2.5-1.5B zero-shot",
        "Qwen2.5-3B zero-shot",
        "Qwen2.5-1.5B QLoRA epoch 2"
    ],
    "input_budgets": {
        "GreekBERT": "256 model tokens",
        "Qwen zero-shot": "1200 characters",
        "QLoRA": "800 characters"
    },
    "timing_requirements": [
        "One model loaded at a time",
        "Explicit GPU warm-up",
        "CUDA synchronization",
        "Batch size 1",
        "Exclude model loading and downloads",
        "Document exact inference and timing scope",
        "Record peak PyTorch allocated VRAM",
        "Record hardware and software versions"
    ],
    "limitations": [
        "Different model input budgets",
        "Different model inference architectures",
        "Results are specific to this workload and GPU",
        "Benchmark does not measure production API latency"
    ]
}

# ------------------------------------
# 6. Save without overwriting changes
# ------------------------------------

manifest_path = (
    BENCHMARK_DIR / "protocol.json"
)

content = (
    json.dumps(
        protocol,
        ensure_ascii=False,
        indent=2,
        sort_keys=True
    ) + "\n"
).encode("utf-8")

if manifest_path.exists():

    assert manifest_path.read_bytes() == content, (
        "Existing benchmark protocol differs"
    )

    print("Existing protocol verified.")

else:

    manifest_path.write_bytes(content)

    print("New protocol saved.")

# ------------------------------------
# 7. Summary
# ------------------------------------

print("\nBENCHMARK CONFIGURATION")
print("-" * 45)

print("Seed:", SEED)
print("Warm-up samples:", len(warmup_data))
print("Measured samples:", len(benchmark_data))
print("Batch size:", BATCH_SIZE)

print("\nDATASET")
print("-" * 45)

print("Validation articles:", len(val_data))
print("SHA256:", validation_sha256)

print("\nOUTPUT")
print("-" * 45)

print("Manifest:", manifest_path)

print("\nOK — benchmark protocol frozen")

New protocol saved.

BENCHMARK CONFIGURATION
---------------------------------------------
Seed: 42
Warm-up samples: 3
Measured samples: 64
Batch size: 1

DATASET
---------------------------------------------
Validation articles: 787
SHA256: d124de2fa8273aed325b80cb30f78adf677c876cee32797a4ca112929dfe22fb

OUTPUT
---------------------------------------------
Manifest: /content/drive/MyDrive/greek-news-nlp/results/inference_benchmark/protocol.json

OK — benchmark protocol frozen


In [ ]:
# CELL 5 — Load Fine-Tuned GreekBERT on GPU

import torch
import unicodedata

from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

# ------------------------------------
# 1. Model configuration
# ------------------------------------

GREEKBERT_DIR = (
    DATA_DIR / "greek-bert-news-classifier"
)

MAX_GREEKBERT_TOKENS = 256

assert GREEKBERT_DIR.is_dir()
assert (GREEKBERT_DIR / "config.json").is_file()
assert (GREEKBERT_DIR / "model.safetensors").is_file()

assert torch.cuda.is_available()

device = torch.device("cuda:0")

# ------------------------------------
# 2. Load tokenizer
# ------------------------------------

print("Loading GreekBERT tokenizer...")

greekbert_tokenizer = AutoTokenizer.from_pretrained(
    str(GREEKBERT_DIR),
    local_files_only=True,
    use_fast=True,
)

# ------------------------------------
# 3. Load fine-tuned model
# ------------------------------------

print("Loading GreekBERT checkpoint...")

greekbert_model = (
    AutoModelForSequenceClassification.from_pretrained(
        str(GREEKBERT_DIR),
        local_files_only=True,
    )
)

greekbert_model = greekbert_model.to(device)
greekbert_model.eval()

# ------------------------------------
# 4. Verify model configuration
# ------------------------------------

num_labels = greekbert_model.config.num_labels

assert num_labels == 17, (
    f"Expected 17 labels, got {num_labels}"
)

assert next(
    greekbert_model.parameters()
).device.type == "cuda"

print("\nMODEL INFORMATION")
print("-" * 50)

print("Model type:", greekbert_model.config.model_type)
print("Model class:", type(greekbert_model).__name__)
print("Number of labels:", num_labels)
print("Device:", next(greekbert_model.parameters()).device)

# ------------------------------------
# 5. Historical GreekBERT preprocessing
# ------------------------------------

def preprocess_greekbert(text):

    text = unicodedata.normalize(
        "NFD",
        text,
    )

    text = "".join(
        char
        for char in text
        if unicodedata.category(char) != "Mn"
    )

    return text.lower()


# ------------------------------------
# 6. Tokenization sanity check
# ------------------------------------

sample_text = preprocess_greekbert(
    benchmark_data[0]["text_clean"]
)

sample_tokens = greekbert_tokenizer(
    sample_text,
    truncation=True,
    max_length=MAX_GREEKBERT_TOKENS,
    return_tensors="pt",
)

assert sample_tokens["input_ids"].shape[0] == 1

assert (
    sample_tokens["input_ids"].shape[1]
    <= MAX_GREEKBERT_TOKENS
)

print("\nTOKENIZATION CHECK")
print("-" * 50)

print("Max token budget:", MAX_GREEKBERT_TOKENS)
print(
    "Sample token count:",
    sample_tokens["input_ids"].shape[1],
)

print(
    "Preprocessing:",
    "NFD + remove Mn + lowercase"
)

# ------------------------------------
# 7. GPU memory after model loading
# ------------------------------------

torch.cuda.synchronize()

allocated_gb = (
    torch.cuda.memory_allocated()
    / 1024**3
)

print("\nGPU MEMORY")
print("-" * 50)

print(
    "Allocated after loading:",
    f"{allocated_gb:.3f} GB",
)

print("\nOK — GreekBERT loaded on GPU")

Loading GreekBERT tokenizer...
Loading GreekBERT checkpoint...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


MODEL INFORMATION
--------------------------------------------------
Model type: bert
Model class: BertForSequenceClassification
Number of labels: 17
Device: cuda:0

TOKENIZATION CHECK
--------------------------------------------------
Max token budget: 256
Sample token count: 225
Preprocessing: NFD + remove Mn + lowercase

GPU MEMORY
--------------------------------------------------
Allocated after loading: 0.421 GB

OK — GreekBERT loaded on GPU


In [ ]:
# CELL 6 — GreekBERT Controlled Latency and VRAM Benchmark

import time
import numpy as np
import pandas as pd
import torch

# ------------------------------------
# 1. Validate benchmark setup
# ------------------------------------

assert torch.cuda.is_available()
assert len(warmup_data) == 3
assert len(benchmark_data) == 64
assert BATCH_SIZE == 1

assert greekbert_model.training is False

DEVICE = torch.device("cuda:0")

print("MODEL:", type(greekbert_model).__name__)
print("DEVICE:", DEVICE)
print("DTYPE:", next(greekbert_model.parameters()).dtype)
print("MEASURED ARTICLES:", len(benchmark_data))
print("WARM-UP ARTICLES:", len(warmup_data))


# ------------------------------------
# 2. Input preparation
# ------------------------------------

def prepare_greekbert_input(text):

    normalized_text = preprocess_greekbert(text)

    encoded = greekbert_tokenizer(
        normalized_text,
        truncation=True,
        max_length=MAX_GREEKBERT_TOKENS,
        return_tensors="pt",
    )

    return encoded


# ------------------------------------
# 3. Warm-up
# ------------------------------------

print("\nRunning GPU warm-up...")

greekbert_model.eval()

with torch.inference_mode():

    for record in warmup_data:

        encoded = prepare_greekbert_input(
            record["text_clean"]
        )

        inputs = {
            key: value.to(DEVICE)
            for key, value in encoded.items()
        }

        outputs = greekbert_model(**inputs)

        _ = outputs.logits.argmax(dim=-1).item()

        torch.cuda.synchronize()

        del encoded, inputs, outputs

torch.cuda.synchronize()

print("Warm-up completed.")


# ------------------------------------
# 4. Reset GPU peak-memory statistics
# ------------------------------------

baseline_allocated = (
    torch.cuda.memory_allocated(DEVICE)
)

torch.cuda.reset_peak_memory_stats(DEVICE)

print(
    "\nBaseline allocated VRAM:",
    f"{baseline_allocated / 1024**2:.2f} MiB"
)


# ------------------------------------
# 5. Measured inference
# ------------------------------------

measurement_rows = []

print("\nStarting measured inference...")

with torch.inference_mode():

    for i, record in enumerate(benchmark_data):

        text = record["text_clean"]

        # Synchronize before starting request timing
        torch.cuda.synchronize()

        total_start = time.perf_counter()

        # Preprocessing and tokenization
        encoded = prepare_greekbert_input(text)

        token_count = int(
            encoded["input_ids"].shape[1]
        )

        # Transfer inputs to GPU
        inputs = {
            key: value.to(DEVICE)
            for key, value in encoded.items()
        }

        # Synchronize before forward timing
        torch.cuda.synchronize()

        forward_start = time.perf_counter()

        # GPU forward pass
        outputs = greekbert_model(**inputs)

        # Wait for GPU computation to finish
        torch.cuda.synchronize()

        forward_end = time.perf_counter()

        # Prediction extraction
        predicted_class = int(
            outputs.logits.argmax(dim=-1).item()
        )

        torch.cuda.synchronize()

        total_end = time.perf_counter()

        # Convert to milliseconds
        forward_ms = (
            forward_end - forward_start
        ) * 1000

        end_to_end_ms = (
            total_end - total_start
        ) * 1000

        measurement_rows.append({
            "document_id": record["document_id"],
            "token_count": token_count,
            "forward_ms": forward_ms,
            "end_to_end_ms": end_to_end_ms,
            "predicted_class_id": predicted_class,
        })

        del encoded, inputs, outputs

        if (i + 1) % 16 == 0:
            print(
                f"Measured {i + 1}/{len(benchmark_data)} articles"
            )

torch.cuda.synchronize()


# ------------------------------------
# 6. GPU peak memory
# ------------------------------------

peak_allocated = (
    torch.cuda.max_memory_allocated(DEVICE)
)

peak_reserved = (
    torch.cuda.max_memory_reserved(DEVICE)
)

additional_peak = max(
    0,
    peak_allocated - baseline_allocated
)


# ------------------------------------
# 7. Aggregate latency statistics
# ------------------------------------

greekbert_timings = pd.DataFrame(
    measurement_rows
)

assert len(greekbert_timings) == 64

assert greekbert_timings["document_id"].is_unique

assert greekbert_timings["token_count"].between(
    1, MAX_GREEKBERT_TOKENS
).all()

assert (
    greekbert_timings["forward_ms"] > 0
).all()

assert (
    greekbert_timings["end_to_end_ms"] >=
    greekbert_timings["forward_ms"]
).all()


def summarize_latency(values):

    values = np.asarray(values, dtype=float)

    return {
        "mean_ms": float(np.mean(values)),
        "median_ms": float(np.median(values)),
        "p90_ms": float(np.percentile(values, 90)),
        "p95_ms": float(np.percentile(values, 95)),
        "min_ms": float(np.min(values)),
        "max_ms": float(np.max(values)),
    }


forward_stats = summarize_latency(
    greekbert_timings["forward_ms"]
)

end_to_end_stats = summarize_latency(
    greekbert_timings["end_to_end_ms"]
)


# ------------------------------------
# 8. Report results
# ------------------------------------

print("\n" + "=" * 55)
print("GREEKBERT — CONTROLLED INFERENCE BENCHMARK")
print("=" * 55)

print("\nFORWARD LATENCY (ms)")

for key, value in forward_stats.items():
    print(f"{key}: {value:.3f}")


print("\nEND-TO-END LATENCY (ms)")

for key, value in end_to_end_stats.items():
    print(f"{key}: {value:.3f}")


print("\nGPU MEMORY")

print(
    "Baseline allocated:",
    f"{baseline_allocated / 1024**2:.2f} MiB"
)

print(
    "Peak allocated:",
    f"{peak_allocated / 1024**2:.2f} MiB"
)

print(
    "Additional peak:",
    f"{additional_peak / 1024**2:.2f} MiB"
)

print(
    "Peak reserved:",
    f"{peak_reserved / 1024**2:.2f} MiB"
)


print("\nINPUT STATISTICS")

print(
    "Average tokens:",
    f"{greekbert_timings['token_count'].mean():.2f}"
)

print(
    "Maximum tokens:",
    greekbert_timings["token_count"].max()
)


print("\nOK — GreekBERT benchmark completed")
print("Results currently stored in memory.")

MODEL: BertForSequenceClassification
DEVICE: cuda:0
DTYPE: torch.float32
MEASURED ARTICLES: 64
WARM-UP ARTICLES: 3

Running GPU warm-up...
Warm-up completed.

Baseline allocated VRAM: 440.44 MiB

Starting measured inference...
Measured 16/64 articles
Measured 32/64 articles
Measured 48/64 articles
Measured 64/64 articles

GREEKBERT — CONTROLLED INFERENCE BENCHMARK

FORWARD LATENCY (ms)
mean_ms: 33.393
median_ms: 34.362
p90_ms: 44.091
p95_ms: 49.233
min_ms: 11.526
max_ms: 61.661

END-TO-END LATENCY (ms)
mean_ms: 42.679
median_ms: 42.615
p90_ms: 54.010
p95_ms: 59.021
min_ms: 13.227
max_ms: 82.877

GPU MEMORY
Baseline allocated: 440.44 MiB
Peak allocated: 448.70 MiB
Additional peak: 8.26 MiB
Peak reserved: 492.00 MiB

INPUT STATISTICS
Average tokens: 211.12
Maximum tokens: 256

OK — GreekBERT benchmark completed
Results currently stored in memory.


In [ ]:
# CELL 7 — Save GreekBERT Benchmark Results

import json
import hashlib
import sys
from pathlib import Path
from importlib import metadata

# ------------------------------------
# 1. Output directory
# ------------------------------------

RESULTS_DIR = (
    DATA_DIR / "results" / "inference_benchmark"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TIMINGS_PATH = RESULTS_DIR / "greekbert_timings.csv"
MANIFEST_PATH = RESULTS_DIR / "greekbert_manifest.json"

# ------------------------------------
# 2. Verify benchmark data
# ------------------------------------

assert len(greekbert_timings) == 64

assert greekbert_timings["document_id"].is_unique

assert greekbert_timings["document_id"].tolist() == [
    row["document_id"]
    for row in benchmark_data
]

assert all(
    greekbert_timings["forward_ms"] > 0
)

assert all(
    greekbert_timings["end_to_end_ms"] >=
    greekbert_timings["forward_ms"]
)

# Verify the frozen protocol.
protocol_path = RESULTS_DIR / "protocol.json"

assert protocol_path.is_file()

saved_protocol = json.loads(
    protocol_path.read_text(encoding="utf-8")
)

assert saved_protocol["measured_document_ids"] == (
    greekbert_timings["document_id"].tolist()
)

assert saved_protocol["validation_sha256"] == (
    validation_sha256
)

print("Benchmark data verified.")

# ------------------------------------
# 3. Prepare timing CSV
# ------------------------------------

timings_bytes = greekbert_timings.to_csv(
    index=False,
    lineterminator="\n",
    float_format="%.8f",
).encode("utf-8")

timings_sha256 = hashlib.sha256(
    timings_bytes
).hexdigest()

# ------------------------------------
# 4. File hashing helper
# ------------------------------------

def sha256_file(path):

    hasher = hashlib.sha256()

    with open(path, "rb") as file:

        for chunk in iter(
            lambda: file.read(4 * 1024 * 1024),
            b"",
        ):
            hasher.update(chunk)

    return hasher.hexdigest()


# ------------------------------------
# 5. Record model checkpoint hashes
# ------------------------------------

print("Calculating model file hashes...")

model_files = [
    "model.safetensors",
    "config.json",
    "tokenizer.json",
    "tokenizer_config.json",
]

model_hashes = {}

for filename in model_files:

    file_path = GREEKBERT_DIR / filename

    if file_path.is_file():

        model_hashes[filename] = sha256_file(
            file_path
        )

        print("Verified:", filename)

assert "model.safetensors" in model_hashes
assert "config.json" in model_hashes

# ------------------------------------
# 6. Software versions
# ------------------------------------

def get_version(package):

    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return "not_installed"


software_versions = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "transformers": get_version("transformers"),
    "accelerate": get_version("accelerate"),
    "numpy": get_version("numpy"),
    "pandas": get_version("pandas"),
    "safetensors": get_version("safetensors"),
}

# ------------------------------------
# 7. Build reproducibility manifest
# ------------------------------------

benchmark_manifest = {

    "benchmark": "controlled_inference_benchmark_v1",

    "model": {
        "name": "GreekBERT fine-tuned",
        "model_type": greekbert_model.config.model_type,
        "checkpoint_directory": str(GREEKBERT_DIR),
        "checkpoint_sha256": model_hashes,
        "num_labels": 17,
        "dtype": str(
            next(greekbert_model.parameters()).dtype
        ),
        "max_length": MAX_GREEKBERT_TOKENS,
        "preprocessing": (
            "Unicode NFD, remove Mn, lowercase"
        ),
    },

    "dataset": {
        "split": "validation",
        "dataset_sha256": validation_sha256,
        "warmup_articles": len(warmup_data),
        "measured_articles": len(benchmark_data),
        "batch_size": BATCH_SIZE,
        "seed": SEED,
    },

    "hardware": {
        "device": torch.cuda.get_device_name(0),
        "cuda_version": torch.version.cuda,
        "total_vram_gib": (
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3
        ),
    },

    "timing_protocol": {
        "warmup": True,
        "cuda_synchronization": True,
        "model_loading_excluded": True,
        "batch_size": 1,
        "forward_scope": (
            "Model forward invocation, synchronized "
            "before and after GPU execution."
        ),
        "end_to_end_scope": (
            "Preprocessing, tokenization, transfer, "
            "model forward, synchronization, "
            "and prediction extraction."
        ),
    },

    "forward_latency_ms": forward_stats,

    "end_to_end_latency_ms": end_to_end_stats,

    "gpu_memory_mib": {
        "baseline_allocated": (
            baseline_allocated / 1024**2
        ),
        "peak_allocated": (
            peak_allocated / 1024**2
        ),
        "additional_peak": (
            additional_peak / 1024**2
        ),
        "peak_reserved": (
            peak_reserved / 1024**2
        ),
    },

    "input_statistics": {
        "mean_tokens": float(
            greekbert_timings["token_count"].mean()
        ),
        "max_tokens": int(
            greekbert_timings["token_count"].max()
        ),
    },

    "software_versions": software_versions,

    "results": {
        "timings_file": "greekbert_timings.csv",
        "timings_sha256": timings_sha256,
    },

    "limitations": [
        "Single GPU benchmark session.",
        "Batch size 1 with 64 validation examples.",
        "Model inputs limited to 256 tokens.",
        "Timing includes explicit CUDA synchronization.",
        "End-to-end latency is not production API latency.",
        "Peak allocated VRAM measures PyTorch allocations.",
        "Checkpoint identity with the historical training "
        "run has not been established.",
        "Different model input budgets limit direct "
        "cross-model comparisons.",
    ],
}

manifest_bytes = (
    json.dumps(
        benchmark_manifest,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    ) + "\n"
).encode("utf-8")

# ------------------------------------
# 8. Save without accidental overwrite
# ------------------------------------

files_to_save = [
    (TIMINGS_PATH, timings_bytes),
    (MANIFEST_PATH, manifest_bytes),
]

for path, content in files_to_save:

    if path.exists():

        assert path.read_bytes() == content, (
            f"Existing file differs: {path}"
        )

        print("Existing file verified:", path.name)

    else:

        path.write_bytes(content)

        print("Saved:", path.name)

    assert path.read_bytes() == content

# ------------------------------------
# 9. Final summary
# ------------------------------------

print("\nGREEKBERT BENCHMARK SAVED")
print("-" * 50)

print("Samples:", len(greekbert_timings))

print(
    "Mean forward latency:",
    f"{forward_stats['mean_ms']:.3f} ms"
)

print(
    "Mean end-to-end latency:",
    f"{end_to_end_stats['mean_ms']:.3f} ms"
)

print(
    "Peak allocated VRAM:",
    f"{peak_allocated / 1024**2:.2f} MiB"
)

print("\nFiles:")
print(TIMINGS_PATH)
print(MANIFEST_PATH)

print("\nOK — GreekBERT results saved")

Benchmark data verified.
Calculating model file hashes...
Verified: model.safetensors
Verified: config.json
Verified: tokenizer.json
Verified: tokenizer_config.json
Saved: greekbert_timings.csv
Saved: greekbert_manifest.json

GREEKBERT BENCHMARK SAVED
--------------------------------------------------
Samples: 64
Mean forward latency: 33.393 ms
Mean end-to-end latency: 42.679 ms
Peak allocated VRAM: 448.70 MiB

Files:
/content/drive/MyDrive/greek-news-nlp/results/inference_benchmark/greekbert_timings.csv
/content/drive/MyDrive/greek-news-nlp/results/inference_benchmark/greekbert_manifest.json

OK — GreekBERT results saved


In [ ]:
# CELL 8 — GreekBERT GPU Cleanup

import gc
import torch
import json
import hashlib

# ------------------------------------
# 1. Verify saved results
# ------------------------------------

assert TIMINGS_PATH.is_file()
assert MANIFEST_PATH.is_file()

saved_manifest = json.loads(
    MANIFEST_PATH.read_text(encoding="utf-8")
)

saved_sha = hashlib.sha256(
    TIMINGS_PATH.read_bytes()
).hexdigest()

assert saved_sha == (
    saved_manifest["results"]["timings_sha256"]
)

print("GreekBERT saved results verified.")

# ------------------------------------
# 2. Memory before cleanup
# ------------------------------------

torch.cuda.synchronize()

before_allocated = (
    torch.cuda.memory_allocated(0) / 1024**2
)

before_reserved = (
    torch.cuda.memory_reserved(0) / 1024**2
)

print("\nBEFORE CLEANUP")
print(f"Allocated: {before_allocated:.2f} MiB")
print(f"Reserved: {before_reserved:.2f} MiB")

# ------------------------------------
# 3. Remove GreekBERT from memory
# ------------------------------------

del greekbert_model
del greekbert_tokenizer

gc.collect()

torch.cuda.empty_cache()
torch.cuda.synchronize()

# ------------------------------------
# 4. Memory after cleanup
# ------------------------------------

after_allocated = (
    torch.cuda.memory_allocated(0) / 1024**2
)

after_reserved = (
    torch.cuda.memory_reserved(0) / 1024**2
)

print("\nAFTER CLEANUP")
print(f"Allocated: {after_allocated:.2f} MiB")
print(f"Reserved: {after_reserved:.2f} MiB")

print(
    "\nReleased allocated:",
    f"{before_allocated - after_allocated:.2f} MiB"
)

# ------------------------------------
# 5. Final verification
# ------------------------------------

assert "greekbert_model" not in globals()
assert "greekbert_tokenizer" not in globals()

if after_allocated > 10:
    print(
        "\nWARNING — Some GPU tensors remain allocated."
    )
else:
    print("\nOK — GPU cleanup completed")

print("CUDA available:", torch.cuda.is_available())

GreekBERT saved results verified.

BEFORE CLEANUP
Allocated: 440.44 MiB
Reserved: 492.00 MiB

AFTER CLEANUP
Allocated: 9.12 MiB
Reserved: 22.00 MiB

Released allocated: 431.32 MiB

OK — GPU cleanup completed
CUDA available: True


In [ ]:
# CELL 9 — Load Qwen2.5-1.5B Zero-Shot on GPU

import torch
import gc

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

from huggingface_hub import HfApi

# ------------------------------------
# 1. Model configuration
# ------------------------------------

QWEN_15B_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

QWEN_DTYPE = torch.float16
QWEN_MAX_CHARS = 1200

DEVICE = torch.device("cuda:0")

assert torch.cuda.is_available()

assert "greekbert_model" not in globals()

# ------------------------------------
# 2. Verify available GPU memory
# ------------------------------------

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

memory_before = (
    torch.cuda.memory_allocated(DEVICE)
    / 1024**2
)

print("GPU memory before loading:")
print(f"{memory_before:.2f} MiB")

# ------------------------------------
# 3. Resolve exact model revision
# ------------------------------------

print("\nResolving model revision...")

QWEN_15B_REVISION = HfApi().model_info(
    repo_id=QWEN_15B_MODEL_ID
).sha

print("Model:", QWEN_15B_MODEL_ID)
print("Revision:", QWEN_15B_REVISION)

# ------------------------------------
# 4. Load tokenizer
# ------------------------------------

print("\nLoading Qwen tokenizer...")

qwen_15b_tokenizer = AutoTokenizer.from_pretrained(
    QWEN_15B_MODEL_ID,
    revision=QWEN_15B_REVISION,
    trust_remote_code=False,
)

# ------------------------------------
# 5. Load model in FP16
# ------------------------------------

print("\nLoading Qwen2.5-1.5B model...")

qwen_15b_model = AutoModelForCausalLM.from_pretrained(
    QWEN_15B_MODEL_ID,
    revision=QWEN_15B_REVISION,
    dtype=QWEN_DTYPE,
    low_cpu_mem_usage=True,
    trust_remote_code=False,
)

# ------------------------------------
# 6. Transfer to GPU
# ------------------------------------

print("\nTransferring model to Tesla T4...")

qwen_15b_model = qwen_15b_model.to(DEVICE)

qwen_15b_model.eval()

torch.cuda.synchronize()

# ------------------------------------
# 7. Verify configuration
# ------------------------------------

model_device = next(
    qwen_15b_model.parameters()
).device

model_dtype = next(
    qwen_15b_model.parameters()
).dtype

assert model_device.type == "cuda"

assert model_dtype == torch.float16

assert qwen_15b_model.config.model_type == "qwen2"

assert not qwen_15b_model.training

num_parameters = sum(
    p.numel()
    for p in qwen_15b_model.parameters()
)

print("\nMODEL INFORMATION")
print("-" * 50)

print("Model:", QWEN_15B_MODEL_ID)
print("Architecture:", type(qwen_15b_model).__name__)
print("Device:", model_device)
print("Dtype:", model_dtype)
print("Parameters:", f"{num_parameters:,}")
print("Input text budget:", QWEN_MAX_CHARS, "characters")

# ------------------------------------
# 8. Tokenizer sanity check
# ------------------------------------

test_encoding = qwen_15b_tokenizer(
    "Αυτή είναι μια δοκιμαστική ελληνική πρόταση.",
    return_tensors="pt",
)

assert test_encoding["input_ids"].shape[0] == 1

print("\nTOKENIZER")
print("-" * 50)

print("Tokenizer:", type(qwen_15b_tokenizer).__name__)

print(
    "Test tokens:",
    test_encoding["input_ids"].shape[1]
)

print("Vocabulary size:", len(qwen_15b_tokenizer))

# ------------------------------------
# 9. GPU memory after loading
# ------------------------------------

torch.cuda.synchronize()

memory_after = (
    torch.cuda.memory_allocated(DEVICE)
    / 1024**2
)

reserved_after = (
    torch.cuda.memory_reserved(DEVICE)
    / 1024**2
)

print("\nGPU MEMORY")
print("-" * 50)

print(
    "Allocated after loading:",
    f"{memory_after:.2f} MiB"
)

print(
    "Reserved after loading:",
    f"{reserved_after:.2f} MiB"
)

print(
    "Additional allocated:",
    f"{memory_after - memory_before:.2f} MiB"
)

print("\nOK — Qwen2.5-1.5B loaded on GPU")

GPU memory before loading:
9.12 MiB

Resolving model revision...


Model: Qwen/Qwen2.5-1.5B-Instruct
Revision: 989aa7980e4cf806f80c7fef2b1adb7bc71aa306

Loading Qwen tokenizer...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Transferring model to Tesla T4...

MODEL INFORMATION
--------------------------------------------------
Model: Qwen/Qwen2.5-1.5B-Instruct
Architecture: Qwen2ForCausalLM
Device: cuda:0
Dtype: torch.float16
Parameters: 1,543,714,304
Input text budget: 1200 characters

TOKENIZER
--------------------------------------------------
Tokenizer: Qwen2Tokenizer
Test tokens: 38
Vocabulary size: 151665

GPU MEMORY
--------------------------------------------------
Allocated after loading: 2955.03 MiB
Reserved after loading: 3156.00 MiB
Additional allocated: 2945.90 MiB

OK — Qwen2.5-1.5B loaded on GPU


In [ ]:
# CELL 10 — Inspect Original Zero-Shot Implementation

import json
from pathlib import Path
from urllib.request import urlopen

# 1. Locate original Notebook 05
local_candidates = [
    DATA_DIR / "05_llm_benchmark.ipynb",
    Path("/content/05_llm_benchmark.ipynb"),
]

notebook_path = next(
    (p for p in local_candidates if p.is_file()),
    None
)

if notebook_path is not None:
    notebook_bytes = notebook_path.read_bytes()
    source_location = str(notebook_path)
else:
    url = (
        "https://raw.githubusercontent.com/"
        "P-Tilemachos/Greek-News-NLP-API/"
        "main/notebooks/05_llm_benchmark.ipynb"
    )

    print("Loading Notebook 05 from GitHub...")

    with urlopen(url, timeout=60) as response:
        notebook_bytes = response.read()

    source_location = url

notebook05 = json.loads(notebook_bytes)

assert notebook05["nbformat"] == 4

print("Source:", source_location)
print("Total notebook cells:", len(notebook05["cells"]))

# 2. Extract code cells
code_cells = []

for index, cell in enumerate(notebook05["cells"]):
    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))
    code_cells.append((index, source))

# 3. Find code related to zero-shot inference
matches = []

for index, source in code_cells:

    lower = source.lower()
    score = 0

    if "apply_chat_template" in lower:
        score += 4

    if "logits" in lower and "label" in lower:
        score += 5

    if "1200" in lower and "prompt" in lower:
        score += 4

    if "token_ids" in lower:
        score += 3

    if "zero_shot" in lower:
        score += 2

    if "candidate" in lower and "token" in lower:
        score += 3

    if score > 0:
        matches.append((score, index, source))

matches.sort(key=lambda item: (-item[0], item[1]))

print("\nRelevant code cells:", len(matches))

# 4. Display top matches for inspection
for score, index, source in matches[:5]:

    print("\n" + "=" * 65)
    print(f"NOTEBOOK 05 — CELL {index + 1}")
    print("Relevance score:", score)
    print("=" * 65)

    print(source[:6000])

    if len(source) > 6000:
        print("\n[OUTPUT TRUNCATED]")

print("\nOK — original zero-shot code inspected")

Loading Notebook 05 from GitHub...
Source: https://raw.githubusercontent.com/P-Tilemachos/Greek-News-NLP-API/main/notebooks/05_llm_benchmark.ipynb
Total notebook cells: 29

Relevant code cells: 16

NOTEBOOK 05 — CELL 27
Relevance score: 11
for n in ["qmodel", "base_inf", "gb"]:
    if n in globals():
        del globals()[n]
gc.collect(); torch.cuda.empty_cache()

def zs_prompt(t, text):
    messages = [
        {"role": "system", "content": "You are a precise news topic classifier."},
        {"role": "user", "content":
            f"Classify the following Greek news article into exactly one of these topics:\n"
            f"{label_list}\n\nArticle:\n{text[:MAX_CHARS]}\n\n"
            f"Answer with the topic name only, exactly as written in the list."},
    ]
    return t.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def zero_shot_first_token(model_id, texts, bs=8):
    t = AutoTokenizer.from_pretrained(model_id); t.padding_side = "left"
    m = AutoModel

In [ ]:
# CELL 11 — Inspect Original Zero-Shot Label Configuration

import re

# ------------------------------------
# 1. Find original variable definitions
# ------------------------------------

patterns = {
    "LABELS": r"^\s*LABELS\s*=",
    "MAX_CHARS": r"^\s*MAX_CHARS\s*=",
    "label_list": r"^\s*label_list\s*=",
    "TEXT_COL": r"^\s*TEXT_COL\s*=",
    "LABEL_COL": r"^\s*LABEL_COL\s*=",
}

found = set()

for index, cell in enumerate(notebook05["cells"]):

    if cell["cell_type"] != "code":
        continue

    source = "".join(cell.get("source", []))
    lines = source.splitlines()

    for name, pattern in patterns.items():

        for line_number, line in enumerate(lines):

            if re.search(pattern, line):

                found.add(name)

                print("\n" + "=" * 55)
                print(f"VARIABLE: {name}")
                print(f"Notebook 05 Cell: {index + 1}")
                print("=" * 55)

                start = max(0, line_number - 2)
                end = min(len(lines), line_number + 8)

                print("\n".join(lines[start:end]))

                break

# ------------------------------------
# 2. Check dataset labels
# ------------------------------------

dataset_labels = sorted({
    record["GPT-IPTC-label"]
    for record in val_data
})

print("\n" + "=" * 55)
print("VALIDATION DATASET LABELS")
print("=" * 55)

for index, label in enumerate(dataset_labels, start=1):
    print(f"{index:02d}. {label}")

print("\nNumber of unique labels:", len(dataset_labels))

assert len(dataset_labels) == 17

# ------------------------------------
# 3. Configuration discovery
# ------------------------------------

print("\n" + "=" * 55)
print("CONFIGURATION DISCOVERY")
print("=" * 55)

for name in patterns:
    print(
        f"{name}:",
        "FOUND" if name in found else "NOT FOUND"
    )

print("\nQwen currently loaded:",
      "qwen_15b_model" in globals())

print(
    "Model device:",
    next(qwen_15b_model.parameters()).device
)

print("\nOK — original label configuration inspected")


VARIABLE: LABELS
Notebook 05 Cell: 3
if N_SAMPLES:
    test = test.sample(N_SAMPLES, random_state=42).reset_index(drop=True)
LABELS = sorted(test[LABEL_COL].astype(str).unique())
print(len(test), "samples,", len(LABELS), "labels")
print(LABELS)

VARIABLE: MAX_CHARS
Notebook 05 Cell: 3
TEXT_COL, LABEL_COL = "text_clean", "GPT-IPTC-label"
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MAX_CHARS = 1200
N_SAMPLES = 100        # smoke test, μετά βάλε None για όλα τα 788
BATCH = 8

test = pd.read_json(TEST_PATH, lines=True)
if N_SAMPLES:
    test = test.sample(N_SAMPLES, random_state=42).reset_index(drop=True)
LABELS = sorted(test[LABEL_COL].astype(str).unique())

VARIABLE: label_list
Notebook 05 Cell: 5
import time

label_list = "\n".join(f"- {l}" for l in LABELS)

def build_prompt(text):
    messages = [
        {"role": "system", "content": "You are a precise news topic classifier."},
        {"role": "user", "content":
            f"Classify the following Greek news article into exactly one of

In [ ]:
# CELL 12 — Qwen2.5-1.5B Zero-Shot Inference Smoke Test

import torch

# ------------------------------------
# 1. Reconstruct original label order
# ------------------------------------

QWEN_LABELS = sorted({
    str(record["GPT-IPTC-label"])
    for record in val_data
})

assert len(QWEN_LABELS) == 17

QWEN_LABEL_LIST = "\n".join(
    f"- {label}"
    for label in QWEN_LABELS
)

assert QWEN_MAX_CHARS == 1200

# ------------------------------------
# 2. Original first-token scoring
# ------------------------------------

first_token_ids = [
    qwen_15b_tokenizer.encode(
        label,
        add_special_tokens=False
    )[0]
    for label in QWEN_LABELS
]

assert len(first_token_ids) == 17

assert len(set(first_token_ids)) == 17, (
    "Duplicate first-token IDs detected"
)

first_token_ids_t = torch.tensor(
    first_token_ids,
    dtype=torch.long,
    device=DEVICE
)

print("LABEL TOKEN VERIFICATION")
print("-" * 55)

for label, token_id in zip(
    QWEN_LABELS,
    first_token_ids
):
    token_text = qwen_15b_tokenizer.decode(
        [token_id]
    )

    print(
        f"{label} -> "
        f"{token_id} ({token_text!r})"
    )

# ------------------------------------
# 3. Original Notebook 05 prompt
# ------------------------------------

def build_qwen_zero_shot_prompt(text):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a precise news topic classifier."
            )
        },
        {
            "role": "user",
            "content": (
                "Classify the following Greek news "
                "article into exactly one of these topics:\n"
                f"{QWEN_LABEL_LIST}\n\n"
                f"Article:\n{text[:QWEN_MAX_CHARS]}\n\n"
                "Answer with the topic name only, "
                "exactly as written in the list."
            )
        }
    ]

    return qwen_15b_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

# ------------------------------------
# 4. Select one benchmark article
# ------------------------------------

sample = benchmark_data[0]

sample_id = sample["document_id"]
sample_text = sample["text_clean"]
true_label = sample["GPT-IPTC-label"]

assert true_label in QWEN_LABELS
assert sample_text.strip()

prompt = build_qwen_zero_shot_prompt(
    sample_text
)

# Match Notebook 05 tokenization behavior.
encoded = qwen_15b_tokenizer(
    prompt,
    return_tensors="pt"
)

input_tokens = int(
    encoded["input_ids"].shape[1]
)

encoded = encoded.to(DEVICE)

print("\nINPUT INFORMATION")
print("-" * 55)

print("Document ID:", sample_id)
print("Article characters:", len(sample_text))
print("Used article characters:", len(sample_text[:1200]))
print("Prompt tokens:", input_tokens)
print("Device:", DEVICE)

# ------------------------------------
# 5. First-token classification
# ------------------------------------

print("\nRunning zero-shot inference...")

qwen_15b_model.eval()

with torch.inference_mode():

    outputs = qwen_15b_model(
        **encoded,
        logits_to_keep=1
    )

    last_logits = outputs.logits[:, -1, :]

    candidate_logits = last_logits[
        :,
        first_token_ids_t
    ]

    best_index = int(
        candidate_logits.argmax(dim=-1).item()
    )

    prediction = QWEN_LABELS[best_index]

torch.cuda.synchronize()

# ------------------------------------
# 6. Inspect prediction
# ------------------------------------

print("\nZERO-SHOT PREDICTION")
print("-" * 55)

print("True label:", true_label)
print("Predicted label:", prediction)

print(
    "Match:",
    prediction == true_label
)

# ------------------------------------
# 7. Top-3 candidate labels
# ------------------------------------

top_values, top_indices = torch.topk(
    candidate_logits[0],
    k=3
)

print("\nTOP-3 CANDIDATE LABELS")
print("-" * 55)

for rank, (idx, score) in enumerate(
    zip(
        top_indices.tolist(),
        top_values.tolist()
    ),
    start=1
):

    print(
        f"{rank}. {QWEN_LABELS[idx]} "
        f"| logit={score:.4f}"
    )

# ------------------------------------
# 8. Final validation
# ------------------------------------

assert last_logits.shape[0] == 1

assert candidate_logits.shape == (1, 17)

assert torch.isfinite(
    candidate_logits
).all()

assert prediction in QWEN_LABELS

print("\nOK — Qwen zero-shot inference working")

LABEL TOKEN VERIFICATION
-------------------------------------------------------
arts, culture, entertainment and media -> 7038 ('arts')
conflict, war and peace -> 6135 ('conf')
crime, law and justice -> 84354 ('crime')
disaster, accident and emergency incident -> 4243 ('dis')
economy, business and finance -> 68 ('e')
education -> 36738 ('education')
environment -> 23294 ('environment')
health -> 12120 ('health')
human interest -> 25312 ('human')
labour -> 14380 ('lab')
lifestyle and leisure -> 75 ('l')
politics -> 9896 ('pol')
religion -> 3748 ('rel')
science and technology -> 39557 ('science')
society -> 82 ('s')
sport -> 60901 ('sport')
weather -> 15206 ('weather')

INPUT INFORMATION
-------------------------------------------------------
Document ID: macocu.el.13384630
Article characters: 864
Used article characters: 864
Prompt tokens: 843
Device: cuda:0

Running zero-shot inference...

ZERO-SHOT PREDICTION
-------------------------------------------------------
True label: economy

In [ ]:
# CELL 13 — Qwen2.5-1.5B Controlled Inference Benchmark

import time
import gc
import numpy as np
import pandas as pd
import torch

# ------------------------------------
# 1. Validate benchmark setup
# ------------------------------------

assert torch.cuda.is_available()
assert len(warmup_data) == 3
assert len(benchmark_data) == 64
assert BATCH_SIZE == 1

assert not qwen_15b_model.training

assert next(
    qwen_15b_model.parameters()
).dtype == torch.float16

assert QWEN_MAX_CHARS == 1200
assert len(first_token_ids) == 17

print("MODEL:", QWEN_15B_MODEL_ID)
print("DEVICE:", DEVICE)
print("DTYPE:", next(qwen_15b_model.parameters()).dtype)
print("MEASURED ARTICLES:", len(benchmark_data))
print("WARM-UP ARTICLES:", len(warmup_data))


# ------------------------------------
# 2. Release previous smoke-test tensors
# ------------------------------------

for name in (
    "encoded",
    "outputs",
    "last_logits",
    "candidate_logits",
    "top_values",
    "top_indices",
    "test_encoding",
):
    if name in globals():
        del globals()[name]

gc.collect()

torch.cuda.empty_cache()
torch.cuda.synchronize()


# ------------------------------------
# 3. Input preparation
# ------------------------------------

def prepare_qwen_15b_input(text):

    prompt = build_qwen_zero_shot_prompt(text)

    encoded = qwen_15b_tokenizer(
        prompt,
        return_tensors="pt",
    )

    return encoded


# ------------------------------------
# 4. GPU warm-up
# ------------------------------------

print("\nRunning GPU warm-up...")

qwen_15b_model.eval()

with torch.inference_mode():

    for record in warmup_data:

        encoded = prepare_qwen_15b_input(
            record["text_clean"]
        )

        inputs = {
            key: value.to(DEVICE)
            for key, value in encoded.items()
        }

        outputs = qwen_15b_model(
            **inputs,
            logits_to_keep=1,
        )

        candidate_scores = outputs.logits[
            0, -1, first_token_ids_t
        ]

        prediction_index = int(
            candidate_scores.argmax().item()
        )

        assert 0 <= prediction_index < 17

        torch.cuda.synchronize()

        del encoded, inputs, outputs, candidate_scores

torch.cuda.synchronize()

gc.collect()

print("Warm-up completed.")


# ------------------------------------
# 5. Reset peak GPU memory statistics
# ------------------------------------

baseline_allocated = torch.cuda.memory_allocated(
    DEVICE
)

torch.cuda.reset_peak_memory_stats(DEVICE)

print(
    "\nBaseline allocated VRAM:",
    f"{baseline_allocated / 1024**2:.2f} MiB"
)


# ------------------------------------
# 6. Controlled inference measurement
# ------------------------------------

measurement_rows = []

print("\nStarting measured inference...")

with torch.inference_mode():

    for i, record in enumerate(benchmark_data):

        text = record["text_clean"]

        # Synchronize before request timing.
        torch.cuda.synchronize()

        total_start = time.perf_counter()

        # Prompt preparation and tokenization.
        encoded = prepare_qwen_15b_input(text)

        token_count = int(
            encoded["input_ids"].shape[1]
        )

        # Transfer to GPU.
        inputs = {
            key: value.to(DEVICE)
            for key, value in encoded.items()
        }

        # Synchronize before forward timing.
        torch.cuda.synchronize()

        forward_start = time.perf_counter()

        # Model forward pass.
        outputs = qwen_15b_model(
            **inputs,
            logits_to_keep=1,
        )

        torch.cuda.synchronize()

        forward_end = time.perf_counter()

        # Original first-token classification.
        candidate_scores = outputs.logits[
            0, -1, first_token_ids_t
        ]

        prediction_index = int(
            candidate_scores.argmax().item()
        )

        prediction = QWEN_LABELS[
            prediction_index
        ]

        torch.cuda.synchronize()

        total_end = time.perf_counter()

        # Timing in milliseconds.
        forward_ms = (
            forward_end - forward_start
        ) * 1000

        end_to_end_ms = (
            total_end - total_start
        ) * 1000

        measurement_rows.append({
            "document_id": record["document_id"],
            "article_chars": len(text),
            "used_article_chars": len(
                text[:QWEN_MAX_CHARS]
            ),
            "token_count": token_count,
            "forward_ms": forward_ms,
            "end_to_end_ms": end_to_end_ms,
            "predicted_label": prediction,
        })

        del encoded, inputs, outputs, candidate_scores

        if (i + 1) % 16 == 0:
            print(
                f"Measured {i + 1}/{len(benchmark_data)} articles"
            )

torch.cuda.synchronize()


# ------------------------------------
# 7. GPU peak memory
# ------------------------------------

peak_allocated = torch.cuda.max_memory_allocated(
    DEVICE
)

peak_reserved = torch.cuda.max_memory_reserved(
    DEVICE
)

additional_peak = max(
    0,
    peak_allocated - baseline_allocated
)


# ------------------------------------
# 8. Validate measurements
# ------------------------------------

qwen_15b_timings = pd.DataFrame(
    measurement_rows
)

assert len(qwen_15b_timings) == 64

assert qwen_15b_timings["document_id"].is_unique

assert qwen_15b_timings["document_id"].tolist() == [
    record["document_id"]
    for record in benchmark_data
]

assert (
    qwen_15b_timings["forward_ms"] > 0
).all()

assert (
    qwen_15b_timings["end_to_end_ms"] >=
    qwen_15b_timings["forward_ms"]
).all()

assert qwen_15b_timings[
    "predicted_label"
].isin(QWEN_LABELS).all()

assert (
    qwen_15b_timings["used_article_chars"]
    <= QWEN_MAX_CHARS
).all()


# ------------------------------------
# 9. Aggregate latency statistics
# ------------------------------------

def summarize_qwen_latency(values):

    values = np.asarray(
        values,
        dtype=float
    )

    return {
        "mean_ms": float(np.mean(values)),
        "median_ms": float(np.median(values)),
        "p90_ms": float(np.percentile(values, 90)),
        "p95_ms": float(np.percentile(values, 95)),
        "min_ms": float(np.min(values)),
        "max_ms": float(np.max(values)),
    }


qwen_15b_forward_stats = summarize_qwen_latency(
    qwen_15b_timings["forward_ms"]
)

qwen_15b_end_to_end_stats = summarize_qwen_latency(
    qwen_15b_timings["end_to_end_ms"]
)


# ------------------------------------
# 10. Report results
# ------------------------------------

print("\n" + "=" * 60)
print("QWEN2.5-1.5B — CONTROLLED INFERENCE BENCHMARK")
print("=" * 60)

print("\nFORWARD LATENCY (ms)")

for key, value in qwen_15b_forward_stats.items():
    print(f"{key}: {value:.3f}")


print("\nEND-TO-END LATENCY (ms)")

for key, value in qwen_15b_end_to_end_stats.items():
    print(f"{key}: {value:.3f}")


print("\nGPU MEMORY")

print(
    "Baseline allocated:",
    f"{baseline_allocated / 1024**2:.2f} MiB"
)

print(
    "Peak allocated:",
    f"{peak_allocated / 1024**2:.2f} MiB"
)

print(
    "Additional peak:",
    f"{additional_peak / 1024**2:.2f} MiB"
)

print(
    "Peak reserved:",
    f"{peak_reserved / 1024**2:.2f} MiB"
)


print("\nINPUT STATISTICS")

print(
    "Average prompt tokens:",
    f"{qwen_15b_timings['token_count'].mean():.2f}"
)

print(
    "Maximum prompt tokens:",
    qwen_15b_timings["token_count"].max()
)

print(
    "Average article characters used:",
    f"{qwen_15b_timings['used_article_chars'].mean():.2f}"
)


print("\nOK — Qwen2.5-1.5B benchmark completed")
print("Results currently stored in memory.")

MODEL: Qwen/Qwen2.5-1.5B-Instruct
DEVICE: cuda:0
DTYPE: torch.float16
MEASURED ARTICLES: 64
WARM-UP ARTICLES: 3

Running GPU warm-up...
Warm-up completed.

Baseline allocated VRAM: 2955.03 MiB

Starting measured inference...
Measured 16/64 articles
Measured 32/64 articles
Measured 48/64 articles
Measured 64/64 articles

QWEN2.5-1.5B — CONTROLLED INFERENCE BENCHMARK

FORWARD LATENCY (ms)
mean_ms: 273.456
median_ms: 306.478
p90_ms: 343.087
p95_ms: 349.073
min_ms: 136.011
max_ms: 352.327

END-TO-END LATENCY (ms)
mean_ms: 277.076
median_ms: 311.241
p90_ms: 346.925
p95_ms: 352.203
min_ms: 138.055
max_ms: 355.667

GPU MEMORY
Baseline allocated: 2955.03 MiB
Peak allocated: 3189.26 MiB
Additional peak: 234.23 MiB
Peak reserved: 3774.00 MiB

INPUT STATISTICS
Average prompt tokens: 989.56
Maximum prompt tokens: 1193
Average article characters used: 1002.14

OK — Qwen2.5-1.5B benchmark completed
Results currently stored in memory.


In [ ]:
# CELL 14 — Save Qwen2.5-1.5B Benchmark Results

import json
import hashlib
import sys
import torch
from pathlib import Path
from importlib import metadata

# ------------------------------------
# 1. Output paths
# ------------------------------------

RESULTS_DIR = (
    DATA_DIR / "results" / "inference_benchmark"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TIMINGS_PATH_QWEN = (
    RESULTS_DIR / "qwen_15b_zero_shot_timings.csv"
)

MANIFEST_PATH_QWEN = (
    RESULTS_DIR / "qwen_15b_zero_shot_manifest.json"
)

PROTOCOL_PATH = RESULTS_DIR / "protocol.json"

# ------------------------------------
# 2. Validate benchmark
# ------------------------------------

assert len(qwen_15b_timings) == 64

assert qwen_15b_timings["document_id"].is_unique

assert qwen_15b_timings["document_id"].tolist() == [
    r["document_id"]
    for r in benchmark_data
]

assert (
    qwen_15b_timings["forward_ms"] > 0
).all()

assert (
    qwen_15b_timings["end_to_end_ms"] >=
    qwen_15b_timings["forward_ms"]
).all()

assert qwen_15b_timings[
    "predicted_label"
].isin(QWEN_LABELS).all()

assert PROTOCOL_PATH.is_file()

saved_protocol = json.loads(
    PROTOCOL_PATH.read_text(encoding="utf-8")
)

assert saved_protocol["measured_document_ids"] == (
    qwen_15b_timings["document_id"].tolist()
)

assert saved_protocol["validation_sha256"] == (
    validation_sha256
)

assert QWEN_MAX_CHARS == 1200
assert BATCH_SIZE == 1
assert len(QWEN_LABELS) == 17

assert (
    next(qwen_15b_model.parameters()).dtype
    == torch.float16
)

print("Benchmark data verified.")

# ------------------------------------
# 3. Prepare CSV
# ------------------------------------

timings_bytes = qwen_15b_timings.to_csv(
    index=False,
    lineterminator="\n",
    float_format="%.8f",
).encode("utf-8")

timings_sha256 = hashlib.sha256(
    timings_bytes
).hexdigest()

# ------------------------------------
# 4. Software versions
# ------------------------------------

def package_version(name):
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return "not_installed"

software_versions = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "transformers": package_version("transformers"),
    "accelerate": package_version("accelerate"),
    "numpy": package_version("numpy"),
    "pandas": package_version("pandas"),
    "huggingface_hub": package_version("huggingface-hub"),
}

# ------------------------------------
# 5. Build reproducibility manifest
# ------------------------------------

qwen_manifest = {

    "benchmark": "controlled_inference_benchmark_v1",

    "model": {
        "name": QWEN_15B_MODEL_ID,
        "revision": QWEN_15B_REVISION,
        "architecture": type(
            qwen_15b_model
        ).__name__,
        "dtype": str(
            next(qwen_15b_model.parameters()).dtype
        ),
        "parameters": sum(
            p.numel()
            for p in qwen_15b_model.parameters()
        ),
        "device": str(DEVICE),
    },

    "inference": {
        "mode": "zero_shot_first_token",
        "max_article_characters": QWEN_MAX_CHARS,
        "chat_template": True,
        "candidate_labels": QWEN_LABELS,
        "first_token_ids": first_token_ids,
        "candidate_scoring": (
            "argmax over first-token logits "
            "of the 17 label names"
        ),
        "logits_to_keep": 1,
        "autoregressive_generation": False,
    },

    "dataset": {
        "split": "validation",
        "sha256": validation_sha256,
        "warmup_articles": len(warmup_data),
        "measured_articles": len(benchmark_data),
        "batch_size": BATCH_SIZE,
        "seed": SEED,
    },

    "hardware": {
        "gpu": torch.cuda.get_device_name(0),
        "cuda_build": torch.version.cuda,
        "total_vram_gib": (
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3
        ),
    },

    "timing_protocol": {
        "gpu_warmup": True,
        "cuda_synchronization": True,
        "model_loading_excluded": True,
        "batch_size": 1,
        "forward_scope": (
            "Synchronized model forward pass; "
            "excludes tokenization and "
            "prediction extraction."
        ),
        "end_to_end_scope": (
            "Prompt preparation, tokenization, "
            "GPU transfer, synchronized forward, "
            "and first-token prediction extraction."
        ),
    },

    "forward_latency_ms": qwen_15b_forward_stats,

    "end_to_end_latency_ms": (
        qwen_15b_end_to_end_stats
    ),

    "gpu_memory_mib": {
        "baseline_allocated": (
            baseline_allocated / 1024**2
        ),
        "peak_allocated": (
            peak_allocated / 1024**2
        ),
        "additional_peak": (
            additional_peak / 1024**2
        ),
        "peak_reserved": (
            peak_reserved / 1024**2
        ),
    },

    "input_statistics": {
        "mean_prompt_tokens": float(
            qwen_15b_timings["token_count"].mean()
        ),
        "max_prompt_tokens": int(
            qwen_15b_timings["token_count"].max()
        ),
        "mean_article_characters_used": float(
            qwen_15b_timings[
                "used_article_chars"
            ].mean()
        ),
    },

    "software_versions": software_versions,

    "results": {
        "timings_file": TIMINGS_PATH_QWEN.name,
        "timings_sha256": timings_sha256,
    },

    "limitations": [
        "Single GPU benchmark session.",
        "Batch size 1, 64 validation articles.",
        "Article input truncated to 1200 characters.",
        "Prompt includes all 17 category names.",
        "First-token scoring, not text generation.",
        "Different input budgets and model precisions.",
        "End-to-end timing includes CUDA synchronization.",
        "Not production API request latency.",
        "Peak allocated VRAM is PyTorch memory, "
        "not total process GPU usage.",
    ],
}

manifest_bytes = (
    json.dumps(
        qwen_manifest,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    ) + "\n"
).encode("utf-8")

# ------------------------------------
# 6. Save without accidental overwrite
# ------------------------------------

files_to_save = [
    (TIMINGS_PATH_QWEN, timings_bytes),
    (MANIFEST_PATH_QWEN, manifest_bytes),
]

for path, content in files_to_save:

    if path.exists():

        assert path.read_bytes() == content, (
            f"Existing file differs: {path}"
        )

        print("Existing file verified:", path.name)

    else:

        path.write_bytes(content)

        print("Saved:", path.name)

    assert path.read_bytes() == content

# ------------------------------------
# 7. Summary
# ------------------------------------

print("\nQWEN2.5-1.5B BENCHMARK SAVED")
print("-" * 50)

print("Samples:", len(qwen_15b_timings))

print(
    "Mean forward latency:",
    f"{qwen_15b_forward_stats['mean_ms']:.3f} ms"
)

print(
    "Mean end-to-end latency:",
    f"{qwen_15b_end_to_end_stats['mean_ms']:.3f} ms"
)

print(
    "Peak allocated VRAM:",
    f"{peak_allocated / 1024**2:.2f} MiB"
)

print("\nFiles:")
print(TIMINGS_PATH_QWEN)
print(MANIFEST_PATH_QWEN)

print("\nOK — Qwen2.5-1.5B results saved")

Benchmark data verified.
Existing file verified: qwen_15b_zero_shot_timings.csv
Existing file verified: qwen_15b_zero_shot_manifest.json

QWEN2.5-1.5B BENCHMARK SAVED
--------------------------------------------------
Samples: 64
Mean forward latency: 273.456 ms
Mean end-to-end latency: 277.076 ms
Peak allocated VRAM: 3189.26 MiB

Files:
/content/drive/MyDrive/greek-news-nlp/results/inference_benchmark/qwen_15b_zero_shot_timings.csv
/content/drive/MyDrive/greek-news-nlp/results/inference_benchmark/qwen_15b_zero_shot_manifest.json

OK — Qwen2.5-1.5B results saved


In [ ]:
# CELL 15 — Qwen2.5-1.5B GPU Cleanup

import gc
import json
import hashlib
import torch

# ------------------------------------
# 1. Verify saved benchmark results
# ------------------------------------

assert TIMINGS_PATH_QWEN.is_file()
assert MANIFEST_PATH_QWEN.is_file()

saved_qwen_manifest = json.loads(
    MANIFEST_PATH_QWEN.read_text(encoding="utf-8")
)

actual_sha = hashlib.sha256(
    TIMINGS_PATH_QWEN.read_bytes()
).hexdigest()

expected_sha = saved_qwen_manifest[
    "results"
]["timings_sha256"]

assert actual_sha == expected_sha

assert len(qwen_15b_timings) == 64

print("Qwen2.5-1.5B saved results verified.")

# ------------------------------------
# 2. GPU memory before cleanup
# ------------------------------------

torch.cuda.synchronize()

before_allocated = (
    torch.cuda.memory_allocated(0) / 1024**2
)

before_reserved = (
    torch.cuda.memory_reserved(0) / 1024**2
)

print("\nBEFORE CLEANUP")
print("-" * 45)
print(f"Allocated: {before_allocated:.2f} MiB")
print(f"Reserved: {before_reserved:.2f} MiB")

# ------------------------------------
# 3. Delete model and GPU tensors
# ------------------------------------

gpu_objects = [
    "qwen_15b_model",
    "qwen_15b_tokenizer",
    "first_token_ids_t",
]

for name in gpu_objects:

    if name in globals():
        del globals()[name]
        print("Removed:", name)

# Remove any remaining temporary inference objects
temporary_objects = [
    "encoded",
    "inputs",
    "outputs",
    "candidate_scores",
    "last_logits",
    "candidate_logits",
    "top_values",
    "top_indices",
]

for name in temporary_objects:

    if name in globals():
        del globals()[name]

# ------------------------------------
# 4. Garbage collection and CUDA cache
# ------------------------------------

gc.collect()

torch.cuda.empty_cache()
torch.cuda.synchronize()

# ------------------------------------
# 5. GPU memory after cleanup
# ------------------------------------

after_allocated = (
    torch.cuda.memory_allocated(0) / 1024**2
)

after_reserved = (
    torch.cuda.memory_reserved(0) / 1024**2
)

released_memory = (
    before_allocated - after_allocated
)

print("\nAFTER CLEANUP")
print("-" * 45)

print(f"Allocated: {after_allocated:.2f} MiB")
print(f"Reserved: {after_reserved:.2f} MiB")

print(
    f"\nReleased allocated: {released_memory:.2f} MiB"
)

# ------------------------------------
# 6. Final verification
# ------------------------------------

assert "qwen_15b_model" not in globals()
assert "qwen_15b_tokenizer" not in globals()

print("\nCUDA available:", torch.cuda.is_available())

if after_allocated > 100:
    print(
        "\nWARNING — Significant GPU memory "
        "remains allocated. Investigate before "
        "loading the next model."
    )
else:
    print("\nOK — Qwen2.5-1.5B GPU cleanup completed")

Qwen2.5-1.5B saved results verified.

BEFORE CLEANUP
---------------------------------------------
Allocated: 2955.03 MiB
Reserved: 3774.00 MiB
Removed: qwen_15b_model
Removed: qwen_15b_tokenizer
Removed: first_token_ids_t

AFTER CLEANUP
---------------------------------------------
Allocated: 9.12 MiB
Reserved: 22.00 MiB

Released allocated: 2945.90 MiB

CUDA available: True

OK — Qwen2.5-1.5B GPU cleanup completed


In [ ]:
# CELL 16 — Load Qwen2.5-3B Zero-Shot on GPU

import gc
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
)

from huggingface_hub import HfApi

# ------------------------------------
# 1. Model configuration
# ------------------------------------

QWEN_3B_MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"

QWEN_3B_DTYPE = torch.float16
QWEN_3B_MAX_CHARS = 1200

DEVICE = torch.device("cuda:0")

assert torch.cuda.is_available()

assert "qwen_15b_model" not in globals()
assert "greekbert_model" not in globals()

assert len(QWEN_LABELS) == 17

# ------------------------------------
# 2. Verify available GPU memory
# ------------------------------------

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

memory_before = (
    torch.cuda.memory_allocated(DEVICE)
    / 1024**2
)

total_vram = (
    torch.cuda.get_device_properties(0).total_memory
    / 1024**3
)

print("GPU:", torch.cuda.get_device_name(0))
print(f"Total VRAM: {total_vram:.2f} GiB")
print(f"Allocated before loading: {memory_before:.2f} MiB")

assert memory_before < 100, (
    "GPU cleanup incomplete. "
    "Investigate before loading Qwen 3B."
)

# ------------------------------------
# 3. Resolve exact model revision
# ------------------------------------

print("\nResolving Qwen2.5-3B model revision...")

QWEN_3B_REVISION = HfApi().model_info(
    repo_id=QWEN_3B_MODEL_ID
).sha

assert QWEN_3B_REVISION

print("Model:", QWEN_3B_MODEL_ID)
print("Revision:", QWEN_3B_REVISION)

# ------------------------------------
# 4. Load tokenizer
# ------------------------------------

print("\nLoading Qwen2.5-3B tokenizer...")

qwen_3b_tokenizer = AutoTokenizer.from_pretrained(
    QWEN_3B_MODEL_ID,
    revision=QWEN_3B_REVISION,
    trust_remote_code=False,
)

qwen_3b_tokenizer.padding_side = "left"

# ------------------------------------
# 5. Load model in FP16
# ------------------------------------

print("\nLoading Qwen2.5-3B model in FP16...")

qwen_3b_model = AutoModelForCausalLM.from_pretrained(
    QWEN_3B_MODEL_ID,
    revision=QWEN_3B_REVISION,
    dtype=QWEN_3B_DTYPE,
    low_cpu_mem_usage=True,
    trust_remote_code=False,
)

# ------------------------------------
# 6. Transfer model to GPU
# ------------------------------------

print("\nTransferring model to Tesla T4...")

qwen_3b_model = qwen_3b_model.to(DEVICE)

qwen_3b_model.eval()

torch.cuda.synchronize()

# ------------------------------------
# 7. Verify loaded model
# ------------------------------------

model_device = next(
    qwen_3b_model.parameters()
).device

model_dtype = next(
    qwen_3b_model.parameters()
).dtype

num_parameters = sum(
    parameter.numel()
    for parameter in qwen_3b_model.parameters()
)

assert model_device.type == "cuda"

assert model_dtype == torch.float16

assert qwen_3b_model.config.model_type == "qwen2"

assert not qwen_3b_model.training

print("\nMODEL INFORMATION")
print("-" * 55)

print("Model:", QWEN_3B_MODEL_ID)
print("Architecture:", type(qwen_3b_model).__name__)
print("Device:", model_device)
print("Dtype:", model_dtype)
print("Parameters:", f"{num_parameters:,}")
print("Article budget:", QWEN_3B_MAX_CHARS, "characters")

# ------------------------------------
# 8. Tokenizer sanity check
# ------------------------------------

test_encoding_3b = qwen_3b_tokenizer(
    "Αυτή είναι μια δοκιμαστική ελληνική πρόταση.",
    return_tensors="pt",
)

assert test_encoding_3b["input_ids"].shape[0] == 1

print("\nTOKENIZER")
print("-" * 55)

print("Tokenizer:", type(qwen_3b_tokenizer).__name__)

print(
    "Test tokens:",
    test_encoding_3b["input_ids"].shape[1]
)

print("Vocabulary size:", len(qwen_3b_tokenizer))

# ------------------------------------
# 9. GPU memory after loading
# ------------------------------------

torch.cuda.synchronize()

memory_after = (
    torch.cuda.memory_allocated(DEVICE)
    / 1024**2
)

reserved_after = (
    torch.cuda.memory_reserved(DEVICE)
    / 1024**2
)

print("\nGPU MEMORY")
print("-" * 55)

print(
    "Allocated after loading:",
    f"{memory_after:.2f} MiB"
)

print(
    "Reserved after loading:",
    f"{reserved_after:.2f} MiB"
)

print(
    "Additional allocated:",
    f"{memory_after - memory_before:.2f} MiB"
)

print("\nOK — Qwen2.5-3B loaded on GPU")

GPU: Tesla T4
Total VRAM: 14.56 GiB
Allocated before loading: 9.12 MiB

Resolving Qwen2.5-3B model revision...
Model: Qwen/Qwen2.5-3B-Instruct
Revision: aa8e72537993ba99e69dfaafa59ed015b17504d1

Loading Qwen2.5-3B tokenizer...


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


Transferring model to Tesla T4...

MODEL INFORMATION
-------------------------------------------------------
Model: Qwen/Qwen2.5-3B-Instruct
Architecture: Qwen2ForCausalLM
Device: cuda:0
Dtype: torch.float16
Parameters: 3,085,938,688
Article budget: 1200 characters

TOKENIZER
-------------------------------------------------------
Tokenizer: Qwen2Tokenizer
Test tokens: 38
Vocabulary size: 151665

GPU MEMORY
-------------------------------------------------------
Allocated after loading: 6003.59 MiB
Reserved after loading: 6160.00 MiB
Additional allocated: 5994.46 MiB

OK — Qwen2.5-3B loaded on GPU


In [ ]:
# CELL 17 — Qwen2.5-3B Zero-Shot Smoke Test

import torch

# ------------------------------------
# 1. Validate model and labels
# ------------------------------------

assert torch.cuda.is_available()
assert len(QWEN_LABELS) == 17
assert QWEN_3B_MAX_CHARS == 1200

assert next(
    qwen_3b_model.parameters()
).device.type == "cuda"

assert next(
    qwen_3b_model.parameters()
).dtype == torch.float16

qwen_3b_model.eval()

# ------------------------------------
# 2. Original label ordering
# ------------------------------------

QWEN_3B_LABELS = sorted(QWEN_LABELS)

QWEN_3B_LABEL_LIST = "\n".join(
    f"- {label}"
    for label in QWEN_3B_LABELS
)

assert QWEN_3B_LABELS == QWEN_LABELS

# ------------------------------------
# 3. First-token candidate IDs
# ------------------------------------

qwen_3b_first_token_ids = [
    qwen_3b_tokenizer.encode(
        label,
        add_special_tokens=False
    )[0]
    for label in QWEN_3B_LABELS
]

assert len(qwen_3b_first_token_ids) == 17

assert len(set(qwen_3b_first_token_ids)) == 17, (
    "Duplicate first-token IDs detected"
)

qwen_3b_first_token_ids_t = torch.tensor(
    qwen_3b_first_token_ids,
    dtype=torch.long,
    device=DEVICE,
)

print("LABEL TOKEN VERIFICATION")
print("-" * 55)

for label, token_id in zip(
    QWEN_3B_LABELS,
    qwen_3b_first_token_ids
):
    token_text = qwen_3b_tokenizer.decode(
        [token_id]
    )

    print(
        f"{label} -> {token_id} ({token_text!r})"
    )

# ------------------------------------
# 4. Reproduce Notebook 05 prompt
# ------------------------------------

def build_qwen_3b_zero_shot_prompt(text):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a precise news topic classifier."
            )
        },
        {
            "role": "user",
            "content": (
                "Classify the following Greek news "
                "article into exactly one of these topics:\n"
                f"{QWEN_3B_LABEL_LIST}\n\n"
                f"Article:\n{text[:QWEN_3B_MAX_CHARS]}\n\n"
                "Answer with the topic name only, "
                "exactly as written in the list."
            )
        },
    ]

    return qwen_3b_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

# ------------------------------------
# 5. Select same sample as 1.5B
# ------------------------------------

sample = benchmark_data[0]

sample_id = sample["document_id"]
sample_text = sample["text_clean"]
true_label = sample["GPT-IPTC-label"]

assert true_label in QWEN_3B_LABELS

prompt = build_qwen_3b_zero_shot_prompt(
    sample_text
)

encoded_3b = qwen_3b_tokenizer(
    prompt,
    return_tensors="pt",
)

input_tokens_3b = int(
    encoded_3b["input_ids"].shape[1]
)

encoded_3b = encoded_3b.to(DEVICE)

print("\nINPUT INFORMATION")
print("-" * 55)

print("Document ID:", sample_id)
print("Article characters:", len(sample_text))
print(
    "Used article characters:",
    len(sample_text[:QWEN_3B_MAX_CHARS])
)
print("Prompt tokens:", input_tokens_3b)
print("Device:", DEVICE)

# ------------------------------------
# 6. Zero-shot first-token inference
# ------------------------------------

print("\nRunning Qwen2.5-3B inference...")

with torch.inference_mode():

    outputs_3b = qwen_3b_model(
        **encoded_3b,
        logits_to_keep=1,
    )

    last_logits_3b = outputs_3b.logits[
        :, -1, :
    ]

    candidate_logits_3b = last_logits_3b[
        :,
        qwen_3b_first_token_ids_t,
    ]

    best_index_3b = int(
        candidate_logits_3b.argmax(
            dim=-1
        ).item()
    )

    prediction_3b = QWEN_3B_LABELS[
        best_index_3b
    ]

torch.cuda.synchronize()

# ------------------------------------
# 7. Prediction
# ------------------------------------

print("\nZERO-SHOT PREDICTION")
print("-" * 55)

print("True label:", true_label)
print("Predicted label:", prediction_3b)
print("Match:", prediction_3b == true_label)

# ------------------------------------
# 8. Top-3 candidate labels
# ------------------------------------

top_values_3b, top_indices_3b = torch.topk(
    candidate_logits_3b[0],
    k=3,
)

print("\nTOP-3 CANDIDATE LABELS")
print("-" * 55)

for rank, (idx, score) in enumerate(
    zip(
        top_indices_3b.tolist(),
        top_values_3b.tolist(),
    ),
    start=1,
):
    print(
        f"{rank}. {QWEN_3B_LABELS[idx]} "
        f"| logit={score:.4f}"
    )

# ------------------------------------
# 9. Final verification
# ------------------------------------

assert last_logits_3b.shape[0] == 1
assert candidate_logits_3b.shape == (1, 17)

assert torch.isfinite(
    candidate_logits_3b
).all()

assert prediction_3b in QWEN_3B_LABELS

print("\nOK — Qwen2.5-3B zero-shot inference working")

# Remove temporary GPU tensors
del (
    encoded_3b,
    outputs_3b,
    last_logits_3b,
    candidate_logits_3b,
    top_values_3b,
    top_indices_3b,
)

torch.cuda.empty_cache()

LABEL TOKEN VERIFICATION
-------------------------------------------------------
arts, culture, entertainment and media -> 7038 ('arts')
conflict, war and peace -> 6135 ('conf')
crime, law and justice -> 84354 ('crime')
disaster, accident and emergency incident -> 4243 ('dis')
economy, business and finance -> 68 ('e')
education -> 36738 ('education')
environment -> 23294 ('environment')
health -> 12120 ('health')
human interest -> 25312 ('human')
labour -> 14380 ('lab')
lifestyle and leisure -> 75 ('l')
politics -> 9896 ('pol')
religion -> 3748 ('rel')
science and technology -> 39557 ('science')
society -> 82 ('s')
sport -> 60901 ('sport')
weather -> 15206 ('weather')

INPUT INFORMATION
-------------------------------------------------------
Document ID: macocu.el.13384630
Article characters: 864
Used article characters: 864
Prompt tokens: 843
Device: cuda:0

Running Qwen2.5-3B inference...

ZERO-SHOT PREDICTION
-------------------------------------------------------
True label: econom

In [ ]:
# CELL 18 — Qwen2.5-3B Controlled Latency & VRAM Benchmark

import time
import gc
import numpy as np
import pandas as pd
import torch

# ------------------------------------
# 1. Validate benchmark configuration
# ------------------------------------

assert torch.cuda.is_available()
assert len(warmup_data) == 3
assert len(benchmark_data) == 64
assert BATCH_SIZE == 1

assert QWEN_3B_MAX_CHARS == 1200
assert len(QWEN_3B_LABELS) == 17
assert len(set(qwen_3b_first_token_ids)) == 17

assert not qwen_3b_model.training

assert next(
    qwen_3b_model.parameters()
).device.type == "cuda"

assert next(
    qwen_3b_model.parameters()
).dtype == torch.float16

print("MODEL:", QWEN_3B_MODEL_ID)
print("DEVICE:", DEVICE)
print("DTYPE:", next(qwen_3b_model.parameters()).dtype)
print("MEASURED ARTICLES:", len(benchmark_data))
print("WARM-UP ARTICLES:", len(warmup_data))


# ------------------------------------
# 2. Clean temporary GPU memory
# ------------------------------------

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()


# ------------------------------------
# 3. Input preparation
# ------------------------------------

def prepare_qwen_3b_input(text):

    prompt = build_qwen_3b_zero_shot_prompt(text)

    encoded = qwen_3b_tokenizer(
        prompt,
        return_tensors="pt"
    )

    return encoded


# ------------------------------------
# 4. GPU warm-up
# ------------------------------------

print("\nRunning GPU warm-up...")

qwen_3b_model.eval()

with torch.inference_mode():

    for record in warmup_data:

        encoded = prepare_qwen_3b_input(
            record["text_clean"]
        )

        inputs = {
            key: value.to(DEVICE)
            for key, value in encoded.items()
        }

        outputs = qwen_3b_model(
            **inputs,
            logits_to_keep=1
        )

        candidate_scores = outputs.logits[
            0, -1, qwen_3b_first_token_ids_t
        ]

        prediction_index = int(
            candidate_scores.argmax().item()
        )

        assert 0 <= prediction_index < 17

        torch.cuda.synchronize()

        del encoded, inputs, outputs, candidate_scores

torch.cuda.synchronize()
gc.collect()

print("Warm-up completed.")


# ------------------------------------
# 5. Reset peak memory statistics
# ------------------------------------

qwen_3b_baseline_allocated = (
    torch.cuda.memory_allocated(DEVICE)
)

torch.cuda.reset_peak_memory_stats(DEVICE)

print(
    "\nBaseline allocated VRAM:",
    f"{qwen_3b_baseline_allocated / 1024**2:.2f} MiB"
)


# ------------------------------------
# 6. Controlled inference measurement
# ------------------------------------

qwen_3b_measurement_rows = []

print("\nStarting measured inference...")

with torch.inference_mode():

    for i, record in enumerate(benchmark_data):

        text = record["text_clean"]

        torch.cuda.synchronize()

        total_start = time.perf_counter()

        # Prompt and tokenization
        encoded = prepare_qwen_3b_input(text)

        token_count = int(
            encoded["input_ids"].shape[1]
        )

        # GPU transfer
        inputs = {
            key: value.to(DEVICE)
            for key, value in encoded.items()
        }

        torch.cuda.synchronize()

        forward_start = time.perf_counter()

        # Forward pass
        outputs = qwen_3b_model(
            **inputs,
            logits_to_keep=1
        )

        torch.cuda.synchronize()

        forward_end = time.perf_counter()

        # First-token classification
        candidate_scores = outputs.logits[
            0, -1, qwen_3b_first_token_ids_t
        ]

        prediction_index = int(
            candidate_scores.argmax().item()
        )

        prediction = QWEN_3B_LABELS[
            prediction_index
        ]

        torch.cuda.synchronize()

        total_end = time.perf_counter()

        forward_ms = (
            forward_end - forward_start
        ) * 1000

        end_to_end_ms = (
            total_end - total_start
        ) * 1000

        qwen_3b_measurement_rows.append({
            "document_id": record["document_id"],
            "article_chars": len(text),
            "used_article_chars": len(
                text[:QWEN_3B_MAX_CHARS]
            ),
            "token_count": token_count,
            "forward_ms": forward_ms,
            "end_to_end_ms": end_to_end_ms,
            "predicted_label": prediction,
        })

        del encoded, inputs, outputs, candidate_scores

        if (i + 1) % 16 == 0:

            print(
                f"Measured {i + 1}/{len(benchmark_data)} articles"
            )

torch.cuda.synchronize()


# ------------------------------------
# 7. GPU memory measurements
# ------------------------------------

qwen_3b_peak_allocated = (
    torch.cuda.max_memory_allocated(DEVICE)
)

qwen_3b_peak_reserved = (
    torch.cuda.max_memory_reserved(DEVICE)
)

qwen_3b_additional_peak = max(
    0,
    qwen_3b_peak_allocated - qwen_3b_baseline_allocated
)


# ------------------------------------
# 8. Validate collected measurements
# ------------------------------------

qwen_3b_timings = pd.DataFrame(
    qwen_3b_measurement_rows
)

assert len(qwen_3b_timings) == 64

assert qwen_3b_timings["document_id"].is_unique

assert qwen_3b_timings["document_id"].tolist() == [
    record["document_id"]
    for record in benchmark_data
]

assert (
    qwen_3b_timings["forward_ms"] > 0
).all()

assert (
    qwen_3b_timings["end_to_end_ms"] >=
    qwen_3b_timings["forward_ms"]
).all()

assert qwen_3b_timings[
    "predicted_label"
].isin(QWEN_3B_LABELS).all()

assert (
    qwen_3b_timings["used_article_chars"]
    <= QWEN_3B_MAX_CHARS
).all()


# ------------------------------------
# 9. Calculate latency statistics
# ------------------------------------

def summarize_qwen_3b_latency(values):

    values = np.asarray(values, dtype=float)

    return {
        "mean_ms": float(np.mean(values)),
        "median_ms": float(np.median(values)),
        "p90_ms": float(np.percentile(values, 90)),
        "p95_ms": float(np.percentile(values, 95)),
        "min_ms": float(np.min(values)),
        "max_ms": float(np.max(values)),
    }


qwen_3b_forward_stats = summarize_qwen_3b_latency(
    qwen_3b_timings["forward_ms"]
)

qwen_3b_end_to_end_stats = summarize_qwen_3b_latency(
    qwen_3b_timings["end_to_end_ms"]
)


# ------------------------------------
# 10. Report actual benchmark results
# ------------------------------------

print("\n" + "=" * 60)
print("QWEN2.5-3B — CONTROLLED INFERENCE BENCHMARK")
print("=" * 60)

print("\nFORWARD LATENCY (ms)")

for key, value in qwen_3b_forward_stats.items():
    print(f"{key}: {value:.3f}")


print("\nEND-TO-END LATENCY (ms)")

for key, value in qwen_3b_end_to_end_stats.items():
    print(f"{key}: {value:.3f}")


print("\nGPU MEMORY")

print(
    "Baseline allocated:",
    f"{qwen_3b_baseline_allocated / 1024**2:.2f} MiB"
)

print(
    "Peak allocated:",
    f"{qwen_3b_peak_allocated / 1024**2:.2f} MiB"
)

print(
    "Additional peak:",
    f"{qwen_3b_additional_peak / 1024**2:.2f} MiB"
)

print(
    "Peak reserved:",
    f"{qwen_3b_peak_reserved / 1024**2:.2f} MiB"
)


print("\nINPUT STATISTICS")

print(
    "Average prompt tokens:",
    f"{qwen_3b_timings['token_count'].mean():.2f}"
)

print(
    "Maximum prompt tokens:",
    qwen_3b_timings["token_count"].max()
)

print(
    "Average article characters used:",
    f"{qwen_3b_timings['used_article_chars'].mean():.2f}"
)


print("\nOK — Qwen2.5-3B benchmark completed")
print("Results currently stored in memory.")

MODEL: Qwen/Qwen2.5-3B-Instruct
DEVICE: cuda:0
DTYPE: torch.float16
MEASURED ARTICLES: 64
WARM-UP ARTICLES: 3

Running GPU warm-up...
Warm-up completed.

Baseline allocated VRAM: 6003.59 MiB

Starting measured inference...
Measured 16/64 articles
Measured 32/64 articles
Measured 48/64 articles
Measured 64/64 articles

QWEN2.5-3B — CONTROLLED INFERENCE BENCHMARK

FORWARD LATENCY (ms)
mean_ms: 503.219
median_ms: 551.920
p90_ms: 635.054
p95_ms: 642.078
min_ms: 243.553
max_ms: 647.203

END-TO-END LATENCY (ms)
mean_ms: 510.485
median_ms: 562.383
p90_ms: 642.853
p95_ms: 649.135
min_ms: 247.120
max_ms: 656.608

GPU MEMORY
Baseline allocated: 6003.59 MiB
Peak allocated: 6307.31 MiB
Additional peak: 303.72 MiB
Peak reserved: 6956.00 MiB

INPUT STATISTICS
Average prompt tokens: 989.56
Maximum prompt tokens: 1193
Average article characters used: 1002.14

OK — Qwen2.5-3B benchmark completed
Results currently stored in memory.


In [ ]:
# CELL 19 — Save Qwen2.5-3B Benchmark Results

import json
import hashlib
import sys
from pathlib import Path
from importlib import metadata

# ------------------------------------
# 1. Output paths
# ------------------------------------

RESULTS_DIR = (
    DATA_DIR / "results" / "inference_benchmark"
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)

QWEN_3B_TIMINGS_PATH = (
    RESULTS_DIR / "qwen_3b_zero_shot_timings.csv"
)

QWEN_3B_MANIFEST_PATH = (
    RESULTS_DIR / "qwen_3b_zero_shot_manifest.json"
)

PROTOCOL_PATH = RESULTS_DIR / "protocol.json"


# ------------------------------------
# 2. Validate benchmark and protocol
# ------------------------------------

assert PROTOCOL_PATH.is_file()

protocol = json.loads(
    PROTOCOL_PATH.read_text(encoding="utf-8")
)

assert len(qwen_3b_timings) == 64

assert qwen_3b_timings["document_id"].is_unique

assert qwen_3b_timings["document_id"].tolist() == [
    r["document_id"] for r in benchmark_data
]

assert protocol["measured_document_ids"] == (
    qwen_3b_timings["document_id"].tolist()
)

assert protocol["validation_sha256"] == (
    validation_sha256
)

assert (
    qwen_3b_timings["forward_ms"] > 0
).all()

assert (
    qwen_3b_timings["end_to_end_ms"] >=
    qwen_3b_timings["forward_ms"]
).all()

assert qwen_3b_timings[
    "predicted_label"
].isin(QWEN_3B_LABELS).all()

assert QWEN_3B_MAX_CHARS == 1200
assert BATCH_SIZE == 1

assert next(
    qwen_3b_model.parameters()
).dtype == torch.float16

print("Benchmark data verified.")


# ------------------------------------
# 3. Prepare CSV and checksum
# ------------------------------------

csv_bytes = qwen_3b_timings.to_csv(
    index=False,
    lineterminator="\n",
    float_format="%.8f",
).encode("utf-8")

csv_sha256 = hashlib.sha256(
    csv_bytes
).hexdigest()


# ------------------------------------
# 4. Software versions
# ------------------------------------

def get_package_version(package):
    try:
        return metadata.version(package)
    except metadata.PackageNotFoundError:
        return "not_installed"


software_versions = {
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "transformers": get_package_version("transformers"),
    "accelerate": get_package_version("accelerate"),
    "numpy": get_package_version("numpy"),
    "pandas": get_package_version("pandas"),
    "huggingface_hub": get_package_version("huggingface-hub"),
}


# ------------------------------------
# 5. Build reproducibility manifest
# ------------------------------------

qwen_3b_manifest = {

    "benchmark": "controlled_inference_benchmark_v1",

    "model": {
        "name": QWEN_3B_MODEL_ID,
        "revision": QWEN_3B_REVISION,
        "architecture": type(
            qwen_3b_model
        ).__name__,
        "dtype": str(
            next(qwen_3b_model.parameters()).dtype
        ),
        "parameters": sum(
            p.numel()
            for p in qwen_3b_model.parameters()
        ),
        "device": str(DEVICE),
    },

    "inference": {
        "mode": "zero_shot_first_token",
        "max_article_characters": QWEN_3B_MAX_CHARS,
        "labels": QWEN_3B_LABELS,
        "first_token_ids": qwen_3b_first_token_ids,
        "prompt_source": (
            "notebooks/05_llm_benchmark.ipynb"
        ),
        "chat_template": True,
        "logits_to_keep": 1,
        "autoregressive_generation": False,
    },

    "dataset": {
        "split": "validation",
        "sha256": validation_sha256,
        "warmup_articles": len(warmup_data),
        "measured_articles": len(benchmark_data),
        "batch_size": BATCH_SIZE,
        "seed": SEED,
    },

    "hardware": {
        "gpu": torch.cuda.get_device_name(0),
        "cuda_version": torch.version.cuda,
        "total_vram_gib": (
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3
        ),
    },

    "timing_protocol": {
        "warmup": True,
        "cuda_synchronization": True,
        "model_loading_excluded": True,
        "batch_size": 1,
        "forward_scope": (
            "Synchronized model forward pass."
        ),
        "end_to_end_scope": (
            "Prompt preparation, tokenization, "
            "GPU transfer, synchronized model forward "
            "and first-token prediction extraction."
        ),
    },

    "forward_latency_ms": qwen_3b_forward_stats,

    "end_to_end_latency_ms": (
        qwen_3b_end_to_end_stats
    ),

    "gpu_memory_mib": {
        "baseline_allocated": (
            qwen_3b_baseline_allocated / 1024**2
        ),
        "peak_allocated": (
            qwen_3b_peak_allocated / 1024**2
        ),
        "additional_peak": (
            qwen_3b_additional_peak / 1024**2
        ),
        "peak_reserved": (
            qwen_3b_peak_reserved / 1024**2
        ),
    },

    "input_statistics": {
        "mean_prompt_tokens": float(
            qwen_3b_timings["token_count"].mean()
        ),
        "max_prompt_tokens": int(
            qwen_3b_timings["token_count"].max()
        ),
        "mean_article_chars_used": float(
            qwen_3b_timings[
                "used_article_chars"
            ].mean()
        ),
    },

    "software_versions": software_versions,

    "results": {
        "timings_file": QWEN_3B_TIMINGS_PATH.name,
        "timings_sha256": csv_sha256,
    },

    "limitations": [
        "Single GPU benchmark session.",
        "64 validation articles, batch size 1.",
        "Article truncated to 1200 characters.",
        "First-token classification, not generation.",
        "Not production API latency.",
        "End-to-end timing includes CUDA synchronization.",
        "VRAM is PyTorch allocated memory.",
        "Cross-model input budgets differ.",
    ],
}

manifest_bytes = (
    json.dumps(
        qwen_3b_manifest,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    ) + "\n"
).encode("utf-8")


# ------------------------------------
# 6. Save without overwriting
# ------------------------------------

files_to_save = [
    (QWEN_3B_TIMINGS_PATH, csv_bytes),
    (QWEN_3B_MANIFEST_PATH, manifest_bytes),
]

for path, content in files_to_save:

    if path.exists():

        assert path.read_bytes() == content, (
            f"Existing file differs: {path}"
        )

        print("Existing file verified:", path.name)

    else:

        path.write_bytes(content)

        print("Saved:", path.name)

    assert path.read_bytes() == content


# ------------------------------------
# 7. Final verification
# ------------------------------------

print("\nQWEN2.5-3B BENCHMARK SAVED")
print("-" * 50)

print("Samples:", len(qwen_3b_timings))

print(
    "Mean forward latency:",
    f"{qwen_3b_forward_stats['mean_ms']:.3f} ms"
)

print(
    "Mean end-to-end latency:",
    f"{qwen_3b_end_to_end_stats['mean_ms']:.3f} ms"
)

print(
    "Peak allocated VRAM:",
    f"{qwen_3b_peak_allocated / 1024**2:.2f} MiB"
)

print("\nFiles:")
print(QWEN_3B_TIMINGS_PATH)
print(QWEN_3B_MANIFEST_PATH)

print("\nOK — Qwen2.5-3B results saved")

Benchmark data verified.
Saved: qwen_3b_zero_shot_timings.csv
Saved: qwen_3b_zero_shot_manifest.json

QWEN2.5-3B BENCHMARK SAVED
--------------------------------------------------
Samples: 64
Mean forward latency: 503.219 ms
Mean end-to-end latency: 510.485 ms
Peak allocated VRAM: 6307.31 MiB

Files:
/content/drive/MyDrive/greek-news-nlp/results/inference_benchmark/qwen_3b_zero_shot_timings.csv
/content/drive/MyDrive/greek-news-nlp/results/inference_benchmark/qwen_3b_zero_shot_manifest.json

OK — Qwen2.5-3B results saved


In [ ]:
# CELL 20 — Qwen2.5-3B GPU Cleanup

import gc
import json
import hashlib
import torch

# ------------------------------------
# 1. Verify saved benchmark results
# ------------------------------------

assert QWEN_3B_TIMINGS_PATH.is_file()
assert QWEN_3B_MANIFEST_PATH.is_file()

saved_manifest = json.loads(
    QWEN_3B_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)

actual_sha256 = hashlib.sha256(
    QWEN_3B_TIMINGS_PATH.read_bytes()
).hexdigest()

expected_sha256 = saved_manifest[
    "results"
]["timings_sha256"]

assert actual_sha256 == expected_sha256

assert len(qwen_3b_timings) == 64

print("Qwen2.5-3B saved results verified.")

# ------------------------------------
# 2. Memory before cleanup
# ------------------------------------

torch.cuda.synchronize()

before_allocated = (
    torch.cuda.memory_allocated(0) / 1024**2
)

before_reserved = (
    torch.cuda.memory_reserved(0) / 1024**2
)

print("\nBEFORE CLEANUP")
print("-" * 45)

print(f"Allocated: {before_allocated:.2f} MiB")
print(f"Reserved: {before_reserved:.2f} MiB")

# ------------------------------------
# 3. Remove Qwen2.5-3B model
# ------------------------------------

gpu_objects = [
    "qwen_3b_model",
    "qwen_3b_tokenizer",
    "qwen_3b_first_token_ids_t",
]

for name in gpu_objects:

    if name in globals():

        del globals()[name]

        print("Removed:", name)

# ------------------------------------
# 4. Remove temporary tensors
# ------------------------------------

temporary_objects = [
    "encoded_3b",
    "outputs_3b",
    "last_logits_3b",
    "candidate_logits_3b",
    "top_values_3b",
    "top_indices_3b",
    "encoded",
    "inputs",
    "outputs",
    "candidate_scores",
]

for name in temporary_objects:

    if name in globals():
        del globals()[name]

# ------------------------------------
# 5. Clear CUDA cache
# ------------------------------------

gc.collect()

torch.cuda.empty_cache()
torch.cuda.synchronize()

# ------------------------------------
# 6. Memory after cleanup
# ------------------------------------

after_allocated = (
    torch.cuda.memory_allocated(0) / 1024**2
)

after_reserved = (
    torch.cuda.memory_reserved(0) / 1024**2
)

released_memory = (
    before_allocated - after_allocated
)

print("\nAFTER CLEANUP")
print("-" * 45)

print(f"Allocated: {after_allocated:.2f} MiB")
print(f"Reserved: {after_reserved:.2f} MiB")

print(
    "\nReleased allocated:",
    f"{released_memory:.2f} MiB"
)

# ------------------------------------
# 7. Final verification
# ------------------------------------

assert "qwen_3b_model" not in globals()
assert "qwen_3b_tokenizer" not in globals()

print("\nCUDA available:", torch.cuda.is_available())

if after_allocated > 100:

    print(
        "\nWARNING — Significant GPU memory remains. "
        "Investigate before loading QLoRA."
    )

else:

    print(
        "\nOK — Qwen2.5-3B GPU cleanup completed"
    )

Qwen2.5-3B saved results verified.

BEFORE CLEANUP
---------------------------------------------
Allocated: 6003.59 MiB
Reserved: 6956.00 MiB
Removed: qwen_3b_model
Removed: qwen_3b_tokenizer
Removed: qwen_3b_first_token_ids_t

AFTER CLEANUP
---------------------------------------------
Allocated: 9.12 MiB
Reserved: 22.00 MiB

Released allocated: 5994.46 MiB

CUDA available: True

OK — Qwen2.5-3B GPU cleanup completed


In [ ]:
# CELL 22 — Verify Saved Benchmarks on CPU

from google.colab import drive
from pathlib import Path
import json
import csv
import hashlib

# Mount Google Drive
drive.mount("/content/drive")

RESULTS_DIR = Path(
    "/content/drive/MyDrive/greek-news-nlp/"
    "results/inference_benchmark"
)

assert RESULTS_DIR.is_dir()

models = {
    "GreekBERT": "greekbert",
    "Qwen 1.5B": "qwen_15b_zero_shot",
    "Qwen 3B": "qwen_3b_zero_shot",
}

protocol = json.loads(
    (RESULTS_DIR / "protocol.json").read_text(
        encoding="utf-8"
    )
)

print("SAVED BENCHMARK VERIFICATION")
print("-" * 55)

for model_name, prefix in models.items():

    csv_path = RESULTS_DIR / f"{prefix}_timings.csv"
    json_path = RESULTS_DIR / f"{prefix}_manifest.json"

    assert csv_path.is_file(), csv_path
    assert json_path.is_file(), json_path

    manifest = json.loads(
        json_path.read_text(encoding="utf-8")
    )

    csv_bytes = csv_path.read_bytes()

    actual_hash = hashlib.sha256(
        csv_bytes
    ).hexdigest()

    expected_hash = manifest["results"]["timings_sha256"]

    assert actual_hash == expected_hash

    with csv_path.open(encoding="utf-8") as f:
        rows = list(csv.DictReader(f))

    assert len(rows) == 64
    assert [r["document_id"] for r in rows] == (
        protocol["measured_document_ids"]
    )

    print(f"\n{model_name}: VERIFIED")
    print("Articles:", len(rows))
    print(
        "Mean end-to-end:",
        f"{manifest['end_to_end_latency_ms']['mean_ms']:.3f} ms"
    )
    print(
        "Peak allocated VRAM:",
        f"{manifest['gpu_memory_mib']['peak_allocated']:.2f} MiB"
    )

print("\nOK — All 3 saved benchmarks verified")

Mounted at /content/drive
SAVED BENCHMARK VERIFICATION
-------------------------------------------------------

GreekBERT: VERIFIED
Articles: 64
Mean end-to-end: 42.679 ms
Peak allocated VRAM: 448.70 MiB

Qwen 1.5B: VERIFIED
Articles: 64
Mean end-to-end: 277.076 ms
Peak allocated VRAM: 3189.26 MiB

Qwen 3B: VERIFIED
Articles: 64
Mean end-to-end: 510.485 ms
Peak allocated VRAM: 6307.31 MiB

OK — All 3 saved benchmarks verified


In [ ]:
# CELL 23 — Build Partial Benchmark Comparison (CPU)

import json
import pandas as pd
from pathlib import Path

RESULTS_DIR = Path(
    "/content/drive/MyDrive/greek-news-nlp/"
    "results/inference_benchmark"
)

model_files = {
    "GreekBERT": "greekbert",
    "Qwen2.5-1.5B Zero-Shot": "qwen_15b_zero_shot",
    "Qwen2.5-3B Zero-Shot": "qwen_3b_zero_shot",
}

rows = []

# ------------------------------------
# 1. Load verified manifests
# ------------------------------------

for model_name, prefix in model_files.items():

    path = RESULTS_DIR / f"{prefix}_manifest.json"

    assert path.is_file()

    manifest = json.loads(
        path.read_text(encoding="utf-8")
    )

    forward = manifest["forward_latency_ms"]
    end_to_end = manifest["end_to_end_latency_ms"]
    memory = manifest["gpu_memory_mib"]
    model_info = manifest["model"]

    inputs = manifest["input_statistics"]

    average_tokens = inputs.get(
        "mean_tokens",
        inputs.get("mean_prompt_tokens")
    )

    rows.append({
        "model": model_name,
        "precision": model_info["dtype"],
        "samples": 64,
        "batch_size": 1,
        "mean_forward_ms": forward["mean_ms"],
        "p95_forward_ms": forward["p95_ms"],
        "mean_end_to_end_ms": end_to_end["mean_ms"],
        "p95_end_to_end_ms": end_to_end["p95_ms"],
        "peak_allocated_mib": memory["peak_allocated"],
        "additional_peak_mib": memory["additional_peak"],
        "average_input_tokens": average_tokens,
    })

# ------------------------------------
# 2. Build comparison table
# ------------------------------------

comparison_df = pd.DataFrame(rows)

assert len(comparison_df) == 3
assert comparison_df["model"].is_unique

print("\nCONTROLLED INFERENCE BENCHMARK")
print("=" * 65)

display(
    comparison_df.round(3)
)

# ------------------------------------
# 3. Save without accidental overwrite
# ------------------------------------

comparison_path = (
    RESULTS_DIR / "benchmark_comparison_partial.csv"
)

csv_bytes = comparison_df.to_csv(
    index=False,
    lineterminator="\n",
    float_format="%.8f",
).encode("utf-8")

if comparison_path.exists():

    assert comparison_path.read_bytes() == csv_bytes, (
        "Existing comparison file differs. "
        "Review it before overwriting."
    )

    print("\nExisting comparison verified.")

else:

    comparison_path.write_bytes(csv_bytes)

    print("\nComparison saved.")

# ------------------------------------
# 4. Confirm scope and limitations
# ------------------------------------

print("\nMODELS INCLUDED:", len(comparison_df))
print("MEASURED ARTICLES PER MODEL: 64")
print("BATCH SIZE: 1")

print("\nIMPORTANT LIMITATIONS:")
print("- Different model input budgets.")
print("- Different model architectures and precisions.")
print("- Measurements are not production API latency.")
print("- QLoRA benchmark is still pending.")

print("\nOutput:", comparison_path)

print("\nOK — Partial comparison completed")


CONTROLLED INFERENCE BENCHMARK


,model,precision,samples,batch_size,mean_forward_ms,p95_forward_ms,mean_end_to_end_ms,p95_end_to_end_ms,peak_allocated_mib,additional_peak_mib,average_input_tokens
0,GreekBERT,torch.float32,64,1,33.393,49.233,42.679,59.021,448.700,8.256,211.125
1,Qwen2.5-1.5B Zero-Shot,torch.float16,64,1,273.456,349.073,277.076,352.203,3189.259,234.231,989.562
2,Qwen2.5-3B Zero-Shot,torch.float16,64,1,503.219,642.078,510.485,649.135,6307.307,303.720,989.562



Comparison saved.

MODELS INCLUDED: 3
MEASURED ARTICLES PER MODEL: 64
BATCH SIZE: 1

IMPORTANT LIMITATIONS:
- Different model input budgets.
- Different model architectures and precisions.
- Measurements are not production API latency.
- QLoRA benchmark is still pending.

Output: /content/drive/MyDrive/greek-news-nlp/results/inference_benchmark/benchmark_comparison_partial.csv

OK — Partial comparison completed


In [ ]:
# CELL 24 — Paired Qwen Benchmark Verification (CPU)

import pandas as pd
import numpy as np
from pathlib import Path

RESULTS_DIR = Path(
    "/content/drive/MyDrive/greek-news-nlp/"
    "results/inference_benchmark"
)

# ------------------------------------
# 1. Load measured results
# ------------------------------------

qwen15 = pd.read_csv(
    RESULTS_DIR / "qwen_15b_zero_shot_timings.csv"
)

qwen3 = pd.read_csv(
    RESULTS_DIR / "qwen_3b_zero_shot_timings.csv"
)

assert len(qwen15) == 64
assert len(qwen3) == 64

# ------------------------------------
# 2. Verify identical inputs
# ------------------------------------

assert qwen15["document_id"].tolist() == (
    qwen3["document_id"].tolist()
)

assert qwen15["token_count"].tolist() == (
    qwen3["token_count"].tolist()
)

assert qwen15["used_article_chars"].tolist() == (
    qwen3["used_article_chars"].tolist()
)

print("PAIRED INPUT VERIFICATION")
print("-" * 50)
print("Same article IDs: YES")
print("Same article order: YES")
print("Same token counts: YES")
print("Same input character counts: YES")

# ------------------------------------
# 3. Create paired comparison
# ------------------------------------

paired = pd.DataFrame({
    "document_id": qwen15["document_id"],
    "prompt_tokens": qwen15["token_count"],
    "qwen_15b_ms": qwen15["end_to_end_ms"],
    "qwen_3b_ms": qwen3["end_to_end_ms"],
})

paired["difference_ms"] = (
    paired["qwen_3b_ms"] -
    paired["qwen_15b_ms"]
)

assert np.isfinite(
    paired[[
        "qwen_15b_ms",
        "qwen_3b_ms",
        "difference_ms"
    ]].to_numpy()
).all()

# ------------------------------------
# 4. Compare mean latencies
# ------------------------------------

mean15 = paired["qwen_15b_ms"].mean()
mean3 = paired["qwen_3b_ms"].mean()

latency_ratio = mean3 / mean15

print("\nPAIRED LATENCY COMPARISON")
print("-" * 50)

print(f"Qwen 1.5B mean: {mean15:.3f} ms")
print(f"Qwen 3B mean:   {mean3:.3f} ms")
print(f"Ratio:          {latency_ratio:.3f}x")

print(
    "Mean paired difference:",
    f"{paired['difference_ms'].mean():.3f} ms"
)

print(
    "Median paired difference:",
    f"{paired['difference_ms'].median():.3f} ms"
)

print(
    "Articles where 3B was slower:",
    int((paired["difference_ms"] > 0).sum()),
    "/ 64"
)

# ------------------------------------
# 5. Show sample comparisons
# ------------------------------------

print("\nFIRST 5 PAIRED MEASUREMENTS")

display(
    paired.head(5).round(3)
)

# ------------------------------------
# 6. Final verification
# ------------------------------------

assert mean15 > 0
assert mean3 > 0
assert len(paired) == 64

print("\nOK — Paired Qwen comparison verified")
print("No files modified.")

PAIRED INPUT VERIFICATION
--------------------------------------------------
Same article IDs: YES
Same article order: YES
Same token counts: YES
Same input character counts: YES

PAIRED LATENCY COMPARISON
--------------------------------------------------
Qwen 1.5B mean: 277.076 ms
Qwen 3B mean:   510.485 ms
Ratio:          1.842x
Mean paired difference: 233.409 ms
Median paired difference: 250.202 ms
Articles where 3B was slower: 64 / 64

FIRST 5 PAIRED MEASUREMENTS


,document_id,prompt_tokens,qwen_15b_ms,qwen_3b_ms,difference_ms
0,macocu.el.13384630,843,213.771,381.427,167.657
1,macocu.el.13527947,607,149.126,266.506,117.380
2,macocu.el.15080352,1162,331.760,589.157,257.397
3,macocu.el.6651875,1177,335.765,600.218,264.454
4,macocu.el.4399624,761,186.764,350.386,163.622



OK — Paired Qwen comparison verified
No files modified.


In [ ]:
# CELL 25 — Generate Partial Inference Benchmark Report (CPU)

import json
import csv
import hashlib
import statistics
from pathlib import Path

# ------------------------------------
# 1. Load saved benchmark files
# ------------------------------------

RESULTS_DIR = Path(
    "/content/drive/MyDrive/greek-news-nlp/"
    "results/inference_benchmark"
)

protocol = json.loads(
    (RESULTS_DIR / "protocol.json").read_text(
        encoding="utf-8"
    )
)

models = {
    "GreekBERT": "greekbert",
    "Qwen2.5-1.5B Zero-Shot": "qwen_15b_zero_shot",
    "Qwen2.5-3B Zero-Shot": "qwen_3b_zero_shot",
}

manifests = {}
timings = {}

for name, prefix in models.items():

    csv_path = RESULTS_DIR / f"{prefix}_timings.csv"
    manifest_path = RESULTS_DIR / f"{prefix}_manifest.json"

    manifest = json.loads(
        manifest_path.read_text(encoding="utf-8")
    )

    csv_bytes = csv_path.read_bytes()

    actual_hash = hashlib.sha256(
        csv_bytes
    ).hexdigest()

    assert actual_hash == (
        manifest["results"]["timings_sha256"]
    )

    with csv_path.open(
        encoding="utf-8", newline=""
    ) as f:
        rows = list(csv.DictReader(f))

    assert len(rows) == 64

    assert [r["document_id"] for r in rows] == (
        protocol["measured_document_ids"]
    )

    manifests[name] = manifest
    timings[name] = rows

print("All benchmark files verified.")

# ------------------------------------
# 2. Verify paired Qwen inputs
# ------------------------------------

q15 = timings["Qwen2.5-1.5B Zero-Shot"]
q3 = timings["Qwen2.5-3B Zero-Shot"]

for a, b in zip(q15, q3):

    assert a["document_id"] == b["document_id"]
    assert a["token_count"] == b["token_count"]

    assert (
        a["used_article_chars"] ==
        b["used_article_chars"]
    )

lat15 = [
    float(r["end_to_end_ms"])
    for r in q15
]

lat3 = [
    float(r["end_to_end_ms"])
    for r in q3
]

differences = [
    b - a
    for a, b in zip(lat15, lat3)
]

ratio = statistics.mean(lat3) / statistics.mean(lat15)

slower_count = sum(
    difference > 0
    for difference in differences
)

# ------------------------------------
# 3. Build comparison table
# ------------------------------------

table_rows = []

for name, manifest in manifests.items():

    forward = manifest["forward_latency_ms"]
    end_to_end = manifest["end_to_end_latency_ms"]
    memory = manifest["gpu_memory_mib"]

    stats = manifest["input_statistics"]

    mean_tokens = stats.get(
        "mean_tokens",
        stats.get("mean_prompt_tokens")
    )

    table_rows.append(
        f"| {name} "
        f"| {forward['mean_ms']:.3f} "
        f"| {end_to_end['mean_ms']:.3f} "
        f"| {end_to_end['p95_ms']:.3f} "
        f"| {memory['peak_allocated']:.2f} "
        f"| {mean_tokens:.2f} |"
    )

comparison_table = "\n".join(table_rows)

# ------------------------------------
# 4. Reproducibility details
# ------------------------------------

revision15 = manifests[
    "Qwen2.5-1.5B Zero-Shot"
]["model"]["revision"]

revision3 = manifests[
    "Qwen2.5-3B Zero-Shot"
]["model"]["revision"]

dataset_sha = protocol["validation_sha256"]

# ------------------------------------
# 5. Construct Markdown report
# ------------------------------------

report = f"""# Controlled Inference Benchmark — Partial Report

## Status

**Partial benchmark:** 3 of 4 models completed.

The Qwen2.5-1.5B QLoRA Epoch 2 benchmark is pending
GPU availability. No results have been estimated for it.

## Experimental Setup

- GPU: NVIDIA Tesla T4
- Measured validation articles: 64
- Warm-up articles: 3
- Batch size: 1
- Random seed: 42
- Model loading excluded from latency measurements
- Explicit CUDA synchronization used
- Fixed article identifiers across all models
- Single measured run per model

Validation dataset SHA256:

`{dataset_sha}`

## Model Configurations

| Model | Precision | Input budget | Inference |
|---|---|---|---|
| GreekBERT | FP32 | 256 tokens | 17-class classification |
| Qwen2.5-1.5B | FP16 | 1200 article characters | Zero-shot first-token scoring |
| Qwen2.5-3B | FP16 | 1200 article characters | Zero-shot first-token scoring |

For Qwen, the prompt includes all 17 class names. The
model performs a forward pass and selects the label
with the highest logit at the first token of its name.
No autoregressive text generation is performed.

## Latency and Memory Results

All latency values are in milliseconds (ms).
Peak memory is PyTorch allocated GPU memory in MiB.

| Model | Mean Forward (ms) | Mean End-to-End (ms) | P95 End-to-End (ms) | Peak Allocated VRAM (MiB) | Mean Input Tokens |
|---|---:|---:|---:|---:|---:|
{comparison_table}

End-to-end measurements include preprocessing or
prompt construction, tokenization, input transfer,
model execution, synchronization and prediction
extraction. They are not production API latency.

## Paired Qwen Comparison

The Qwen2.5-1.5B and Qwen2.5-3B benchmarks use
identical article IDs, article order, article
character counts and prompt token counts.

- Mean end-to-end latency ratio (3B / 1.5B): {ratio:.3f}x
- Mean paired latency difference: {statistics.mean(differences):.3f} ms
- Median paired latency difference: {statistics.median(differences):.3f} ms
- Articles where 3B was slower: {slower_count}/64

Both zero-shot models were evaluated using FP16,
a 1200-character article limit and the same
first-token classification methodology.

These values describe the measured sample and
execution environment, not a universal performance
ratio or causal estimate.

## Reproducibility

Qwen2.5-1.5B model revision:

`{revision15}`

Qwen2.5-3B model revision:

`{revision3}`

GreekBERT checkpoint file hashes are stored in
`greekbert_manifest.json`.

The benchmark directory contains:

- `protocol.json`
- Per-model timing CSV files
- Per-model benchmark manifests
- `benchmark_comparison_partial.csv`

Individual CSV files are verified against the SHA256
checksums in their manifests.

## Limitations

1. The benchmark uses one Tesla T4 session and only
   64 selected validation articles per model.
2. There are no repeated timing sessions or
   confidence intervals for latency.
3. GreekBERT and Qwen use different input budgets,
   architectures and precision settings.
4. The Qwen results measure first-token classification,
   not full language-model response generation.
5. End-to-end latency includes explicit CUDA
   synchronization and excludes an actual HTTP API.
6. Peak allocated VRAM measures PyTorch allocations,
   not all GPU memory usage by the runtime.
7. The relationship between the saved GreekBERT
   checkpoint and the historical API checkpoint
   remains unresolved.
8. The QLoRA benchmark is incomplete; the current
   comparison must not be presented as final.

## Remaining Work

- Restore GPU access.
- Verify bitsandbytes NF4 compatibility.
- Load the saved QLoRA Epoch 2 adapter.
- Benchmark the QLoRA model using the frozen protocol.
- Update the comparison and prepare the final report.
"""

# ------------------------------------
# 6. Save safely
# ------------------------------------

REPORT_PATH = (
    RESULTS_DIR / "benchmark_report_partial.md"
)

report_bytes = report.encode("utf-8")

if REPORT_PATH.exists():

    assert REPORT_PATH.read_bytes() == report_bytes, (
        "Existing report differs. "
        "Review before overwriting."
    )

    print("Existing report verified.")

else:

    REPORT_PATH.write_bytes(report_bytes)

    print("Partial report saved.")

# ------------------------------------
# 7. Final verification
# ------------------------------------

assert REPORT_PATH.read_bytes() == report_bytes

print("\nPARTIAL BENCHMARK REPORT")
print("-" * 55)

print("Models:", len(models))
print("Articles per model:", 64)
print("Qwen paired inputs: VERIFIED")
print("QLoRA: PENDING")

print("\nReport:")
print(REPORT_PATH)

print("\nReport size:", len(report_bytes), "bytes")

print("\nOK — Partial benchmark report completed")

All benchmark files verified.
Partial report saved.

PARTIAL BENCHMARK REPORT
-------------------------------------------------------
Models: 3
Articles per model: 64
Qwen paired inputs: VERIFIED
QLoRA: PENDING

Report:
/content/drive/MyDrive/greek-news-nlp/results/inference_benchmark/benchmark_report_partial.md

Report size: 3952 bytes

OK — Partial benchmark report completed


In [ ]:
# CELL 26 — QLoRA Checkpoint Readiness Check (CPU)

from pathlib import Path
import json
import hashlib
import torch

# ------------------------------------
# 1. Paths
# ------------------------------------

DATA_DIR = Path(
    "/content/drive/MyDrive/greek-news-nlp"
)

QLORA_DIR = DATA_DIR / "qlora_qwen2.5-1.5b_ep2"

CONFIG_PATH = QLORA_DIR / "adapter_config.json"
WEIGHTS_PATH = QLORA_DIR / "adapter_model.safetensors"

RESULTS_DIR = DATA_DIR / "results/inference_benchmark"

# ------------------------------------
# 2. Verify files
# ------------------------------------

assert CONFIG_PATH.is_file(), "Adapter config missing"
assert WEIGHTS_PATH.is_file(), "Adapter weights missing"

print("QLORA CHECKPOINT VERIFICATION")
print("-" * 55)

print("Config: FOUND")
print("Weights: FOUND")

# ------------------------------------
# 3. Read adapter configuration
# ------------------------------------

config = json.loads(
    CONFIG_PATH.read_text(encoding="utf-8")
)

assert str(config.get("peft_type")).upper() == "LORA"

print("\nADAPTER CONFIGURATION")
print("-" * 55)

print(
    "Base model:",
    config.get("base_model_name_or_path")
)

print("PEFT type:", config.get("peft_type"))
print("LoRA rank:", config.get("r"))
print("LoRA alpha:", config.get("lora_alpha"))
print("LoRA dropout:", config.get("lora_dropout"))
print("Target modules:", config.get("target_modules"))

# ------------------------------------
# 4. Check file size and SHA256
# ------------------------------------

def calculate_sha256(path):

    hasher = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(
            lambda: f.read(4 * 1024 * 1024),
            b""
        ):
            hasher.update(chunk)

    return hasher.hexdigest()


weights_size = (
    WEIGHTS_PATH.stat().st_size / 1024**2
)

weights_sha256 = calculate_sha256(WEIGHTS_PATH)

print("\nCHECKPOINT INTEGRITY")
print("-" * 55)

print(f"Weights size: {weights_size:.2f} MiB")
print("Weights SHA256:", weights_sha256)

# ------------------------------------
# 5. Check benchmark artifacts
# ------------------------------------

required_files = [
    "protocol.json",
    "greekbert_manifest.json",
    "qwen_15b_zero_shot_manifest.json",
    "qwen_3b_zero_shot_manifest.json",
    "benchmark_comparison_partial.csv",
    "benchmark_report_partial.md",
]

print("\nBENCHMARK ARTIFACTS")
print("-" * 55)

for filename in required_files:

    file_path = RESULTS_DIR / filename

    assert file_path.is_file(), filename

    print("VERIFIED:", filename)

# ------------------------------------
# 6. Check current runtime
# ------------------------------------

print("\nRUNTIME STATUS")
print("-" * 55)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("QLoRA GPU benchmark can be prepared.")
else:
    print("QLoRA GPU benchmark: PENDING")

print("\nOK — QLoRA checkpoint readiness verified")

QLORA CHECKPOINT VERIFICATION
-------------------------------------------------------
Config: FOUND
Weights: FOUND

ADAPTER CONFIGURATION
-------------------------------------------------------
Base model: Qwen/Qwen2.5-1.5B-Instruct
PEFT type: LORA
LoRA rank: 16
LoRA alpha: 32
LoRA dropout: 0.05
Target modules: ['down_proj', 'v_proj', 'k_proj', 'o_proj', 'up_proj', 'q_proj', 'gate_proj']

CHECKPOINT INTEGRITY
-------------------------------------------------------
Weights size: 70.49 MiB
Weights SHA256: b9591ad53c483209a387cecba8ea793516601660032e4c800b0de9ebfc6e56ab

BENCHMARK ARTIFACTS
-------------------------------------------------------
VERIFIED: protocol.json
VERIFIED: greekbert_manifest.json
VERIFIED: qwen_15b_zero_shot_manifest.json
VERIFIED: qwen_3b_zero_shot_manifest.json
VERIFIED: benchmark_comparison_partial.csv
VERIFIED: benchmark_report_partial.md

RUNTIME STATUS
-------------------------------------------------------
PyTorch: 2.11.0+cpu
CUDA available: False
QLoRA GPU b

In [1]:
# CELL 27 — Resume-only QLoRA NF4 GPU benchmark and final comparison
# Run this one cell after cells 1-2; do NOT rerun the existing benchmarks.
# It validates the frozen files before doing any new inference.

from pathlib import Path
from importlib import metadata
from statistics import mean
import csv
import gc
import hashlib
import importlib
import json
import random
import subprocess
import sys
import time

from google.colab import drive
import numpy as np
import pandas as pd
import torch

ROOT = Path('/content/drive/MyDrive/greek-news-nlp')
if not ROOT.exists():
    drive.mount('/content/drive')
assert ROOT.is_dir(), f'Google Drive directory not found: {ROOT}'
assert torch.cuda.is_available(), 'Connect to GPU runtime (Tesla T4) before running.'
DEVICE = torch.device('cuda:0')
OUT = ROOT / 'results' / 'inference_benchmark'
PROTO_PATH = OUT / 'protocol.json'
VAL_PATH = ROOT / 'el_val.jsonl'
ADAPTER_DIR = ROOT / 'qlora_qwen2.5-1.5b_ep2'
ADAPTER_CONFIG = ADAPTER_DIR / 'adapter_config.json'
ADAPTER_WEIGHTS = ADAPTER_DIR / 'adapter_model.safetensors'
MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
Q_PREFIX = 'qwen_15b_qlora_epoch2'
Q_CSV = OUT / (Q_PREFIX + '_timings.csv')
Q_JSON = OUT / (Q_PREFIX + '_manifest.json')
FINAL_CSV = OUT / 'benchmark_comparison_final.csv'
FINAL_REPORT = OUT / 'benchmark_report_final.md'
BASELINE_PREFIXES = {
    'GreekBERT': 'greekbert',
    'Qwen2.5-1.5B Zero-Shot': 'qwen_15b_zero_shot',
    'Qwen2.5-3B Zero-Shot': 'qwen_3b_zero_shot',
}

# ----------------- Step A: frozen inputs and previously saved results -----------------
for path in [PROTO_PATH, VAL_PATH, ADAPTER_CONFIG, ADAPTER_WEIGHTS]:
    assert path.is_file(), f'Missing required input: {path}'
protocol = json.loads(PROTO_PATH.read_text(encoding='utf-8'))
assert protocol['benchmark'] == 'controlled_inference_benchmark_v1'
assert protocol['seed'] == 42
assert protocol['warmup_samples'] == 3
assert protocol['measured_samples'] == 64
assert protocol['batch_size'] == 1
assert hashlib.sha256(VAL_PATH.read_bytes()).hexdigest() == protocol['validation_sha256']
with VAL_PATH.open(encoding='utf-8') as f:
    records = [json.loads(line) for line in f if line.strip()]
assert len(records) == 787
assert len({r['document_id'] for r in records}) == 787
rng = random.Random(42)
indices = list(range(len(records)))
rng.shuffle(indices)
warmup = [records[i] for i in indices[:3]]
measured = [records[i] for i in indices[3:67]]
assert [r['document_id'] for r in warmup] == protocol['warmup_document_ids']
assert [r['document_id'] for r in measured] == protocol['measured_document_ids']
labels = sorted({str(r['GPT-IPTC-label']) for r in records})
assert len(labels) == 17
adapter_config = json.loads(ADAPTER_CONFIG.read_text(encoding='utf-8'))
assert adapter_config['peft_type'].upper() == 'LORA'
assert adapter_config['base_model_name_or_path'] == MODEL_ID


def hash_file(path):
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def read_checked(prefix):
    csv_path = OUT / f'{prefix}_timings.csv'
    json_path = OUT / f'{prefix}_manifest.json'
    assert csv_path.is_file() and json_path.is_file(), f'Missing benchmark: {prefix}'
    meta = json.loads(json_path.read_text(encoding='utf-8'))
    assert hash_file(csv_path) == meta['results']['timings_sha256'], f'Hash mismatch: {prefix}'
    rows = pd.read_csv(csv_path)
    assert len(rows) == 64
    assert rows['document_id'].astype(str).tolist() == [str(r['document_id']) for r in measured]
    assert (rows['forward_ms'] > 0).all()
    assert (rows['end_to_end_ms'] >= rows['forward_ms']).all()
    return meta, rows


verified = {}
for name, prefix in BASELINE_PREFIXES.items():
    verified[name] = read_checked(prefix)[0]
    print('VERIFIED (no rerun):', name)


def safe_save(path, raw_bytes):
    if path.exists():
        assert path.read_bytes() == raw_bytes, f'Conflicting existing output; not overwritten: {path}'
        print('VERIFIED existing:', path.name)
    else:
        path.write_bytes(raw_bytes)
        assert path.read_bytes() == raw_bytes
        print('SAVED:', path.name)


# ----------------- Step B: only the missing QLoRA benchmark -----------------
if Q_CSV.exists() or Q_JSON.exists():
    assert Q_CSV.exists() and Q_JSON.exists(), 'Only one QLoRA artifact exists; inspect manually.'
    qmeta, qdata = read_checked(Q_PREFIX)
    assert qmeta['checkpoint']['adapter_weights_sha256'] == hash_file(ADAPTER_WEIGHTS)
    assert qmeta['dataset']['sha256'] == protocol['validation_sha256']
    assert qmeta['inference']['max_article_characters'] == 800
    print('QLoRA results already exist and are verified — skipping GPU inference.')
else:
    # bitsandbytes was NOT installed in the earlier saved notebook output.
    # A 4-bit NF4 benchmark requires the real NF4 runtime, not an FP16 substitution.
    try:
        import bitsandbytes as bnb
    except ImportError:
        print('Installing bitsandbytes for 4-bit NF4 inference...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes'])
        importlib.invalidate_caches()
        import bitsandbytes as bnb

    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import PeftModel

    # Pin Qwen base to the revision already benchmarked in the saved 1.5B run.
    revision = verified['Qwen2.5-1.5B Zero-Shot']['model']['revision']
    assert isinstance(revision, str) and len(revision) >= 32
    print('Loading QLoRA adapter on the matching Qwen base revision:', revision)

    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=False,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=revision, trust_remote_code=False)
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=revision,
        quantization_config=quantization,
        dtype=torch.float16,
        device_map={'': 0},
        trust_remote_code=False,
    )
    assert getattr(base, 'is_loaded_in_4bit', False), 'Model is not actually in 4-bit.'
    model = PeftModel.from_pretrained(base, str(ADAPTER_DIR), is_trainable=False)
    model.eval()
    assert not model.training
    assert next(model.parameters()).device.type == 'cuda'

    # Transparent, fixed inference protocol: 800 article characters, 17-class
    # name prompt, first-token label scoring. Not free-text generation.
    label_list = '\n'.join(f'- {x}' for x in labels)
    label_ids = [tokenizer.encode(x, add_special_tokens=False)[0] for x in labels]
    assert len(set(label_ids)) == 17, 'First-token label IDs are not unique.'
    label_ids_t = torch.tensor(label_ids, dtype=torch.long, device=DEVICE)

    def prepare(record):
        article = record['text_clean'][:800]
        messages = [
            {'role': 'system', 'content': 'You are a precise news topic classifier.'},
            {'role': 'user', 'content': (
                'Classify the following Greek news article into exactly one of these topics:\n'
                f'{label_list}\n\nArticle:\n{article}\n\n'
                'Answer with the topic name only, exactly as written in the list.'
            )},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        enc = tokenizer(prompt, return_tensors='pt')
        return enc, len(article)

    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    print('Running three warm-up articles...')
    with torch.inference_mode():
        for row in warmup:
            enc, _ = prepare(row)
            inp = {k: v.to(DEVICE) for k, v in enc.items()}
            output = model(**inp, logits_to_keep=1)
            _ = int(output.logits[0, -1, label_ids_t].argmax().item())
            torch.cuda.synchronize()
            del enc, inp, output

    torch.cuda.synchronize()
    gc.collect()
    baseline_mem = torch.cuda.memory_allocated(DEVICE)
    torch.cuda.reset_peak_memory_stats(DEVICE)
    print('Measuring 64 new QLoRA inferences...')
    rows = []
    with torch.inference_mode():
        for i, row in enumerate(measured, start=1):
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            enc, used_chars = prepare(row)
            tokens = int(enc['input_ids'].shape[1])
            inp = {k: v.to(DEVICE) for k, v in enc.items()}
            torch.cuda.synchronize()
            t1 = time.perf_counter()
            output = model(**inp, logits_to_keep=1)
            torch.cuda.synchronize()
            t2 = time.perf_counter()
            pred = labels[int(output.logits[0, -1, label_ids_t].argmax().item())]
            torch.cuda.synchronize()
            t3 = time.perf_counter()
            rows.append({
                'document_id': row['document_id'],
                'article_chars': len(row['text_clean']),
                'used_article_chars': used_chars,
                'token_count': tokens,
                'forward_ms': 1000 * (t2 - t1),
                'end_to_end_ms': 1000 * (t3 - t0),
                'predicted_label': pred,
            })
            del enc, inp, output
            if i % 16 == 0:
                print(f'Measured {i}/64')
    torch.cuda.synchronize()
    peak_mem = torch.cuda.max_memory_allocated(DEVICE)
    peak_reserved = torch.cuda.max_memory_reserved(DEVICE)
    qdata = pd.DataFrame(rows)
    assert len(qdata) == 64 and qdata.document_id.is_unique
    assert qdata.document_id.astype(str).tolist() == [str(r['document_id']) for r in measured]
    assert (qdata.forward_ms > 0).all() and (qdata.end_to_end_ms >= qdata.forward_ms).all()
    assert qdata.predicted_label.isin(labels).all()

    def stats(col):
        x = qdata[col].to_numpy(dtype=float)
        return {k:float(v) for k,v in {
            'mean_ms': np.mean(x), 'median_ms': np.median(x),
            'p90_ms': np.percentile(x, 90), 'p95_ms': np.percentile(x, 95),
            'min_ms': np.min(x), 'max_ms': np.max(x),
        }.items()}

    csv_bytes = qdata.to_csv(index=False, lineterminator='\n', float_format='%.8f').encode('utf-8')
    def version(name):
        try: return metadata.version(name)
        except metadata.PackageNotFoundError: return 'not-installed'

    qmeta = {
        'benchmark': 'controlled_inference_benchmark_v1',
        'model': {'name': MODEL_ID + ' + QLoRA epoch 2', 'revision': revision,
                  'architecture': type(model).__name__, 'dtype': 'NF4 storage; FP16 compute',
                  'device': str(DEVICE)},
        'checkpoint': {'adapter_config': str(ADAPTER_CONFIG.name),
                       'adapter_weights_sha256': hash_file(ADAPTER_WEIGHTS),
                       'adapter_weights_size_bytes': ADAPTER_WEIGHTS.stat().st_size},
        'inference': {
            'mode': 'qlora_nf4_first_token_classification',
            'max_article_characters': 800, 'labels': labels,
            'first_token_ids': label_ids, 'logits_to_keep': 1,
            'chat_template': True, 'autoregressive_generation': False,
            'prompt': '17 label names; classify news article and answer with topic name only',
            'quantization': 'bnb NF4, compute fp16, double_quant=False',
        },
        'dataset': {'split': 'validation', 'sha256': protocol['validation_sha256'],
                    'seed': 42, 'warmup_articles': 3, 'measured_articles': 64, 'batch_size': 1},
        'hardware': {'gpu': torch.cuda.get_device_name(0), 'cuda_version': torch.version.cuda,
                     'total_vram_gib': torch.cuda.get_device_properties(0).total_memory / 1024**3},
        'timing_protocol': {'cuda_synchronization': True, 'model_loading_excluded': True,
                            'forward_scope': 'synchronized single forward pass',
                            'end_to_end_scope': 'prompt + tokenization + transfer + forward + prediction'},
        'forward_latency_ms': stats('forward_ms'),
        'end_to_end_latency_ms': stats('end_to_end_ms'),
        'gpu_memory_mib': {'baseline_allocated': baseline_mem / 1024**2,
                           'peak_allocated': peak_mem / 1024**2,
                           'additional_peak': max(0, peak_mem - baseline_mem) / 1024**2,
                           'peak_reserved': peak_reserved / 1024**2},
        'input_statistics': {'mean_prompt_tokens': float(qdata.token_count.mean()),
                             'max_prompt_tokens': int(qdata.token_count.max()),
                             'mean_article_chars_used': float(qdata.used_article_chars.mean())},
        'software_versions': {'python': sys.version.split()[0], 'torch': torch.__version__,
                              **{x:version(x) for x in ['transformers', 'peft', 'bitsandbytes', 'accelerate',
                                                       'numpy', 'pandas', 'safetensors']}},
        'results': {'timings_file': Q_CSV.name,
                    'timings_sha256': hashlib.sha256(csv_bytes).hexdigest()},
        'limitations': [
            'One GPU session, 64 articles, no repeat-session confidence intervals.',
            'Uses a fixed 17-label first-token prompt, not free text generation.',
            'Input limited to 800 article characters.',
            'Inferred prompt need not reproduce original QLoRA training/evaluation prompt exactly.',
            'Other models use different input limits, architectures and precision.',
            'Not measured over the production HTTP API.',
        ],
    }
    manifest_bytes = (json.dumps(qmeta, sort_keys=True, ensure_ascii=False, indent=2) + '\n').encode('utf-8')
    safe_save(Q_CSV, csv_bytes)
    safe_save(Q_JSON, manifest_bytes)
    del model, base, tokenizer, label_ids_t
    gc.collect()
    torch.cuda.empty_cache()
    print('QLoRA GPU benchmark saved and model unloaded.')

# ----------------- Step C: verify 4 models and write final-only artifacts -----------------
verified['Qwen2.5-1.5B QLoRA NF4'] = read_checked(Q_PREFIX)[0]
assert verified['Qwen2.5-1.5B QLoRA NF4']['checkpoint']['adapter_weights_sha256'] == hash_file(ADAPTER_WEIGHTS)

entries = []
for name, m in verified.items():
    fwd, e2e, mem = m['forward_latency_ms'], m['end_to_end_latency_ms'], m['gpu_memory_mib']
    ins = m['input_statistics']
    entries.append({
        'model': name, 'precision': m['model']['dtype'], 'samples': 64,
        'batch_size': 1, 'mean_forward_ms': fwd['mean_ms'],
        'p95_forward_ms': fwd['p95_ms'], 'mean_end_to_end_ms': e2e['mean_ms'],
        'p95_end_to_end_ms': e2e['p95_ms'], 'peak_allocated_mib': mem['peak_allocated'],
        'additional_peak_mib': mem['additional_peak'],
        'average_input_tokens': ins.get('mean_tokens', ins.get('mean_prompt_tokens')),
    })
comparison = pd.DataFrame(entries)
assert len(comparison) == 4 and comparison.model.is_unique
safe_save(FINAL_CSV, comparison.to_csv(index=False, lineterminator='\n', float_format='%.8f').encode('utf-8'))

hdr = ('| Model | Mean forward (ms) | Mean end-to-end (ms) | P95 E2E (ms) | Peak allocated VRAM (MiB) |\n'
       '|---|---:|---:|---:|---:|')
body = '\n'.join(
    f"| {r['model']} | {r['mean_forward_ms']:.3f} | {r['mean_end_to_end_ms']:.3f} | "
    f"{r['p95_end_to_end_ms']:.3f} | {r['peak_allocated_mib']:.2f} |"
    for r in entries
)
report = (
    '# Controlled Inference Benchmark — Final Four-Model Report\n\n'
    '## Scope\n\n'
    'Four measured model configurations on 64 fixed Greek validation articles, 3 warm-ups, batch size 1, '
    'Tesla T4 GPU. All results use the saved frozen protocol and same article IDs. '
    'Each model has one controlled benchmark run; training and checkpoint loading are excluded.\n\n'
    '## Measured performance\n\n' + hdr + '\n' + body + '\n\n'
    '## Model inference protocols\n\n'
    '- GreekBERT: FP32; max 256 tokens; supervised classification.\n'
    '- Qwen2.5-1.5B and Qwen2.5-3B zero-shot: FP16; max 1200 article characters; first-token 17-label scoring.\n'
    '- Qwen2.5-1.5B QLoRA epoch 2: 4-bit NF4 storage, FP16 compute; max 800 article characters; '
    'saved LoRA adapter; first-token 17-label scoring with an explicitly documented fixed prompt.\n\n'
    '## Interpretation / limitations\n\n'
    '- These are **configuration comparisons**, not matched-input model-size or tuning ablations.\n'
    '- The QLoRA benchmark uses a fixed first-token 17-label inference prompt; '
    'its exact equivalence to the original training/evaluation prompt has not been established.\n'
    '- Only one GPU session per model and 64 articles: do not generalize timings to other hardware or workloads.\n'
    '- Results are **not production HTTP API latency** and do not measure autoregressive answer generation.\n'
    '- Models have different input limits and precision. Peak PyTorch allocated memory excludes some runtime memory.\n'
    '- GreekBERT checkpoint identity relative to the historical API remains unresolved.\n'
    '- This benchmark measures inference speed/VRAM; it does not measure model accuracy or faithfulness.\n\n'
    '## Reproducibility / artifacts\n\n'
    f'- Frozen validation SHA256: `{protocol["validation_sha256"]}`\n'
    f'- QLoRA adapter weights SHA256: `{hash_file(ADAPTER_WEIGHTS)}`\n'
    '- Existing per-model CSV files and manifests remain unchanged.\n'
    '- Four per-model manifest/timing pairs have been checksum-verified.\n'
    '- `benchmark_comparison_partial.csv` and `benchmark_report_partial.md` remain unchanged.\n'
    '- New outputs: `qwen_15b_qlora_epoch2_timings.csv`, `qwen_15b_qlora_epoch2_manifest.json`, '
    '`benchmark_comparison_final.csv`, `benchmark_report_final.md`.\n'
)
safe_save(FINAL_REPORT, report.encode('utf-8'))
print('\nFINAL FOUR-MODEL RESULTS')
print(comparison.round(3).to_string(index=False))
print('\nALL FOUR MODELS VERIFIED. Final report:', FINAL_REPORT)
print('No older benchmark or protocol files were overwritten.')


Mounted at /content/drive
VERIFIED (no rerun): GreekBERT
VERIFIED (no rerun): Qwen2.5-1.5B Zero-Shot
VERIFIED (no rerun): Qwen2.5-3B Zero-Shot
Installing bitsandbytes for 4-bit NF4 inference...
Loading QLoRA adapter on the matching Qwen base revision: 989aa7980e4cf806f80c7fef2b1adb7bc71aa306


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Running three warm-up articles...
Measuring 64 new QLoRA inferences...
Measured 16/64
Measured 32/64
Measured 48/64
Measured 64/64
SAVED: qwen_15b_qlora_epoch2_timings.csv
SAVED: qwen_15b_qlora_epoch2_manifest.json
QLoRA GPU benchmark saved and model unloaded.
SAVED: benchmark_comparison_final.csv
SAVED: benchmark_report_final.md

FINAL FOUR-MODEL RESULTS
                 model                 precision  samples  batch_size  mean_forward_ms  p95_forward_ms  mean_end_to_end_ms  p95_end_to_end_ms  peak_allocated_mib  additional_peak_mib  average_input_tokens
             GreekBERT             torch.float32       64           1           33.393          49.233              42.679             59.021             448.700                8.256               211.125
Qwen2.5-1.5B Zero-Shot             torch.float16       64           1          273.456         349.073             277.076            352.203            3189.259              234.231               989.562
  Qwen2.5-3B Zero-Shot     

In [2]:
from pathlib import Path

RESULTS = Path(
    "/content/drive/MyDrive/greek-news-nlp/results/inference_benchmark"
)

print("=== FINAL REPORT ===")
print((RESULTS / "benchmark_report_final.md").read_text(encoding="utf-8"))

print("\n=== SAVED FILES ===")
for file in sorted(RESULTS.iterdir()):
    if file.is_file():
        print(file.name)

=== FINAL REPORT ===
# Controlled Inference Benchmark — Final Four-Model Report

## Scope

Four measured model configurations on 64 fixed Greek validation articles, 3 warm-ups, batch size 1, Tesla T4 GPU. All results use the saved frozen protocol and same article IDs. Each model has one controlled benchmark run; training and checkpoint loading are excluded.

## Measured performance

| Model | Mean forward (ms) | Mean end-to-end (ms) | P95 E2E (ms) | Peak allocated VRAM (MiB) |
|---|---:|---:|---:|---:|
| GreekBERT | 33.393 | 42.679 | 59.021 | 448.70 |
| Qwen2.5-1.5B Zero-Shot | 273.456 | 277.076 | 352.203 | 3189.26 |
| Qwen2.5-3B Zero-Shot | 503.219 | 510.485 | 649.135 | 6307.31 |
| Qwen2.5-1.5B QLoRA NF4 | 299.936 | 303.439 | 422.657 | 1369.13 |

## Model inference protocols

- GreekBERT: FP32; max 256 tokens; supervised classification.
- Qwen2.5-1.5B and Qwen2.5-3B zero-shot: FP16; max 1200 article characters; first-token 17-label scoring.
- Qwen2.5-1.5B QLoRA epoch 2: 4-bit NF4 stor

In [3]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files

RESULTS = Path(
    "/content/drive/MyDrive/greek-news-nlp/results/inference_benchmark"
)

filenames = [
    "protocol.json",
    "greekbert_manifest.json",
    "greekbert_timings.csv",
    "qwen_15b_zero_shot_manifest.json",
    "qwen_15b_zero_shot_timings.csv",
    "qwen_3b_zero_shot_manifest.json",
    "qwen_3b_zero_shot_timings.csv",
    "qwen_15b_qlora_epoch2_manifest.json",
    "qwen_15b_qlora_epoch2_timings.csv",
    "benchmark_comparison_partial.csv",
    "benchmark_comparison_final.csv",
    "benchmark_report_partial.md",
    "benchmark_report_final.md",
]

zip_path = Path("/content/inference_benchmark_review_bundle.zip")

for name in filenames:
    assert (RESULTS / name).is_file(), f"Missing: {name}"

with ZipFile(zip_path, "w", ZIP_DEFLATED) as z:
    for name in filenames:
        z.write(RESULTS / name, arcname=name)

print(f"ZIP READY: {zip_path.name}")
print(f"Files: {len(filenames)}")
print(f"Size: {zip_path.stat().st_size / 1024:.1f} KB")

files.download(str(zip_path))

ZIP READY: inference_benchmark_review_bundle.zip
Files: 13
Size: 19.6 KB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>